In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install duckdb

# FHIR-SQL Reinforcement Learning -- DAPO, continuing each SFT seed's best checkpoint

Reinforcement learning on top of the SFT models, using DAPO
(Decoupled Clip and Dynamic sAmpling Policy Optimization; Yu et al., 2025 -- see CITATIONS.md).
DAPO is a GRPO-family policy-gradient method: no separate value/critic network, advantage is the
within-group reward z-score across several sampled completions of the same prompt.

**Three of DAPO's four techniques are implemented, one deliberately isn't:**
- **Clip-Higher** -- asymmetric PPO clipping (`eps_low=0.20`, `eps_high=0.28`, matching the
  paper's own reported values), giving low-probability tokens more room to grow than high-probability
  ones get to shrink, which counteracts entropy collapse.
- **Dynamic Sampling** -- prompt-groups where every sampled completion got the same reward
  (std=0, so the group-normalized advantage is exactly zero for all of them) are dropped and
  replaced with newly-sampled prompts, so every gradient step trains on groups that actually
  carry signal.
- **Token-level loss** -- the policy-gradient loss is normalized by total response tokens across
  the whole batch, not by sequence count then averaged, so long and short responses don't get
  arbitrarily different per-token weight.
- **Overlong Reward Shaping is NOT implemented.** That technique exists to discourage runaway-length
  generations in DAPO's original long chain-of-thought math domain, where responses run to
  thousands of tokens. SQL responses here are capped at `max_new_tokens_rollout` and are typically
  a few dozen tokens -- the failure mode that technique targets doesn't apply to this domain.

The core loss/advantage/filtering math (`compute_dapo_loss`, `gather_token_logprobs`,
`group_advantages`, `keep_groups`) is reused as-is; everything around it -- rollout sampling, the
execution-based reward, dynamic-sampling collection, the training loop, checkpointing, multi-seed
driving, and the final SFT-vs-RL comparison -- is new, built to match `sft_train.ipynb`'s shape.

Not yet executed -- no GPU here, and it depends on `sft_outputs/seed_<seed>/best/` existing for
each seed, which requires the SFT run to finish first.

In [ ]:
# ==== environment & paths ====
import os, shutil

# Reduces CUDA-allocator fragmentation from variable-length rollouts (every generated response has
# a different length) -- must be set before torch initializes its CUDA context.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    ON_COLAB = True
except Exception:
    ON_COLAB = False

# RunPod: RUNPOD_POD_ID is set in every RunPod container. Uses a persistent network
# volume (survives pod stop/start) mounted at /workspace, so -- unlike Colab's Drive FUSE mount,
# which has real sync lag -- writes to ROOT here are plain local-disk writes, durable without
# needing to round-trip every checkpoint back to Drive. rclone is only used once, to PULL the
# read-only inputs (model weights, schema, DBs, training data) onto the volume the first time
# it's used.
ON_RUNPOD = 'RUNPOD_POD_ID' in os.environ

if ON_COLAB:
    _default_root = '/content/drive/MyDrive/projects/FHIRSQL'
elif ON_RUNPOD:
    _default_root = '/workspace/FHIRSQL'
else:
    _default_root = 'C:/dev/fhirsql-phase2'
ROOT = os.environ.get('FHIRSQL_ROOT', _default_root)
LOCAL_DIR = '/content/local' if ON_COLAB else ROOT   # Colab needs a real local copy (FUSE read latency);
                                                       # RunPod's persistent volume and this Windows box are
                                                       # both already local disk, no separate copy needed.
RL_OUTPUTS_ROOT = os.path.join(ROOT, 'rl_outputs')   # OUT_BASE (below, once CFG exists) namespaces this by CFG['run_name']
SFT_OUT_BASE = os.path.join(ROOT, 'sft_outputs')   # where each seed's SFT best/ checkpoint lives
os.makedirs(LOCAL_DIR, exist_ok=True)

if ON_RUNPOD:
    import subprocess
    RCLONE_REMOTE = os.environ.get('FHIRSQL_RCLONE_REMOTE', 'gdrive:projects/FHIRSQL')
    if shutil.which('rclone') is None:
        print('[runpod] rclone not found -- installing...')
        subprocess.run('curl https://rclone.org/install.sh | sudo bash', shell=True, check=True)
    remote_name = RCLONE_REMOTE.split(':')[0] + ':'
    remotes = subprocess.run(['rclone', 'listremotes'], capture_output=True, text=True).stdout
    if remote_name not in remotes:
        raise RuntimeError(
            f"rclone has no '{remote_name}' remote configured. This can't be set up non-"
            f"interactively (Google OAuth needs a browser step) -- run `rclone config` once on "
            f"this pod, or copy an already-authorized rclone.conf to ~/.config/rclone/rclone.conf, "
            f"then re-run this cell. See https://rclone.org/drive/ for the one-time setup."
        )
    print(f'[runpod] syncing {RCLONE_REMOTE} -> {ROOT} (skips files that already match -- fast on repeat runs)...')
    subprocess.run(['rclone', 'copy', RCLONE_REMOTE, ROOT, '--progress'], check=True)


def push_to_drive_backup():
    '''Not called automatically -- checkpoints on the RunPod persistent volume are already
    durable, this is purely optional convenience/off-volume backup. Call manually whenever wanted.'''
    if not ON_RUNPOD:
        print('push_to_drive_backup() is only meaningful on RunPod (ROOT already IS the Drive-backed store on Colab)')
        return
    import subprocess
    subprocess.run(['rclone', 'copy', ROOT, os.environ.get('FHIRSQL_RCLONE_REMOTE', 'gdrive:projects/FHIRSQL'), '--progress'], check=True)

DRIVE_JSONL = os.path.join(ROOT, 'data', 'training', 'sft_final_plan.jsonl')
DRIVE_SCHEMA = os.path.join(ROOT, 'schema', 'schema.sql')
DRIVE_DUCKDB = os.path.join(ROOT, 'data', 'train.duckdb')
DRIVE_MODEL_PATH = os.path.join(ROOT, 'Qwen2.5-Coder-14B-Instruct')

# Heldout benchmark: disjoint 6,383-patient population, two concept arms -- familiar (382
# concepts also used in training data, isolates population generalization) and unseen (87
# concepts that never appear anywhere in train, isolates concept generalization). See
# METHODOLOGY_LOG.md 'Evaluation objective' -> 'Heldout benchmark design' for the full rationale.
DRIVE_HELDOUT_DUCKDB = os.path.join(ROOT, 'data', 'heldout.duckdb')
DRIVE_HELDOUT_JSONL = {
    'familiar': os.path.join(ROOT, 'data', 'training', 'heldout_benchmark_plan_familiar.jsonl'),
    'unseen': os.path.join(ROOT, 'data', 'training', 'heldout_benchmark_plan_unseen.jsonl'),
}


def _copy_local(src, dst_name):
    dst = os.path.join(LOCAL_DIR, dst_name)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
    return dst


def _copy_local_dir(src, dst_name):
    dst = os.path.join(LOCAL_DIR, dst_name)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    return dst


LOCAL_JSONL = _copy_local(DRIVE_JSONL, 'sft_final_plan.jsonl')
LOCAL_SCHEMA = _copy_local(DRIVE_SCHEMA, 'schema.sql')
LOCAL_MODEL_PATH = _copy_local_dir(DRIVE_MODEL_PATH, 'Qwen2.5-Coder-14B-Instruct')
LOCAL_DUCKDB = _copy_local(DRIVE_DUCKDB, 'train.duckdb') if ON_COLAB else DRIVE_DUCKDB
LOCAL_HELDOUT_DUCKDB = _copy_local(DRIVE_HELDOUT_DUCKDB, 'heldout.duckdb') if ON_COLAB else DRIVE_HELDOUT_DUCKDB
LOCAL_HELDOUT_JSONL = {
    arm: (_copy_local(p, f'heldout_benchmark_plan_{arm}.jsonl') if ON_COLAB else p)
    for arm, p in DRIVE_HELDOUT_JSONL.items()
}

print('on_colab =', ON_COLAB, '| on_runpod =', ON_RUNPOD)
print('ROOT             =', ROOT)
print('RL_OUTPUTS_ROOT  =', RL_OUTPUTS_ROOT)
print('SFT_OUT_BASE     =', SFT_OUT_BASE)
print('LOCAL_JSONL  =', LOCAL_JSONL)
print('LOCAL_SCHEMA =', LOCAL_SCHEMA)
print('LOCAL_MODEL_PATH =', LOCAL_MODEL_PATH)
print('LOCAL_DUCKDB     =', LOCAL_DUCKDB)
print('LOCAL_HELDOUT_DUCKDB =', LOCAL_HELDOUT_DUCKDB)
print('LOCAL_HELDOUT_JSONL  =', LOCAL_HELDOUT_JSONL)

Mounted at /content/drive
on_colab = True
ROOT             = /content/drive/MyDrive/projects/FHIRSQL
RL_OUTPUTS_ROOT  = /content/drive/MyDrive/projects/FHIRSQL/rl_outputs
SFT_OUT_BASE     = /content/drive/MyDrive/projects/FHIRSQL/sft_outputs
LOCAL_JSONL  = /content/local/sft_final.jsonl
LOCAL_SCHEMA = /content/local/schema.sql
LOCAL_MODEL_PATH = /content/local/Qwen2.5-Coder-14B-Instruct
LOCAL_DUCKDB     = /content/local/train.duckdb
LOCAL_HELDOUT_DUCKDB = /content/local/heldout.duckdb
LOCAL_HELDOUT_JSONL  = {'familiar': '/content/local/heldout_benchmark_familiar.jsonl', 'unseen': '/content/local/heldout_benchmark_unseen.jsonl'}


In [ ]:
# ==== imports ====
import json, random, time, math, sys, functools, contextlib, statistics, gc, re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training, PeftModel
import duckdb

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = (DEVICE == 'cuda')
AMP_DTYPE = torch.bfloat16


def amp_ctx():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if USE_AMP else contextlib.nullcontext()


if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print('GPU:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP bf16 ON | TF32 ON')
print('torch', torch.__version__, '| device', DEVICE)

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | 102 GB | AMP bf16 ON | TF32 ON
torch 2.11.0+cu128 | device cuda


In [ ]:
# ==== CONFIG ====
CFG = dict(
    model_name=LOCAL_MODEL_PATH,
    quant_type='nf4',

    # kept for FHIRSQLLLM/_get_dora() compatibility -- not used on the primary path, since the RL
    # adapter is loaded FROM each seed's SFT best/ checkpoint (already rank=16), not built fresh.
    dora_rank=16, dora_alpha=32, dora_dropout=0.05, dora_bias='none',
    dora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],

    max_seq_len=4096,          # ~1,200-1,500-token schema-dominated prompts + up to 2048 completion
                                # tokens needs headroom past 3072. A char-length-based estimate
                                # (~3.3 chars/token) -- verify against the real tokenizer.
    dev_fraction=0.08, test_fraction=0.08, split_seed=20260804,   # identical split to SFT
    num_workers=12,

    # rollout / dynamic sampling
    group_size=4,                  # G completions sampled per prompt (GRPO/DAPO group)
    rollout_groups_per_batch=8,    # target number of reward-variance>0 groups per training step
    rollout_prompts_per_batch=10,  # fixed directly rather than probed each run. Longer completions
                                    # (max_new_tokens_rollout up to 2048) increase KV-cache memory
                                    # pressure per sequence -- watch peak GPU memory on new hardware.
    max_rollout_attempts=20,       # safety cap: stop resampling even if target isn't reached
    rollout_temperature=1.0, rollout_top_p=1.0,
    max_new_tokens_rollout=2048,    # a ceiling, not a target -- generation stops at EOS, so real
                                    # average cost tracks real average completion length
                                    # (~300-400 tokens), not this cap.
    max_new_tokens_eval=2048,       # explicit now (previously relied on scattered .get(..., N)
                                    # fallback defaults at each call site) -- same value everywhere.

    # reward: correctness is a strict 0/1 gate; efficiency only ever adds a small bonus ON TOP
    # of a correct answer, so a faster wrong query can never outscore a correct one.
    efficiency_bonus_max=0.1,      # max extra reward for a correct query that's faster than gold's
    efficiency_min_timing_ms=0.5,  # floor under which wall-clock timing is dominated by measurement noise

    # Tiered penalty for wrong answers -- a flat 0.0 for every failure mode would mean a group of 4
    # completions failing in DIFFERENT ways still looked like zero
    # reward-variance to Dynamic Sampling and got dropped just like a group that was uniformly
    # correct. Ranked worst-to-least-bad by how much the failure reflects vs. genuine SQL/schema
    # understanding, and -- given this is a clinical-data context -- confidently hallucinating an
    # answer to a genuinely unanswerable question is judged worse than any other failure mode,
    # including broken SQL. All four are well below the >=1.0 floor for any correct answer, so
    # the existing safeguard (a wrong answer can never outscore a correct one) holds regardless
    # of these values -- asserted below, not just assumed.
    reward_wrong_result=0.0,        # SQL executes, wrong result -- unchanged baseline, mildest failure
    reward_non_executing=-0.2,      # SQL doesn't execute at all (syntax/schema error)
    reward_false_abstention=-0.1,   # abstained on a question that WAS answerable
    reward_missed_abstention=-0.3,  # answered (hallucinated SQL) on a question that was genuinely UNANSWERABLE
    reward_hardcoded_value=-0.2,    # SQL executes AND matches gold's result, but hardcodes a
                                     # literal terminology code/system instead of resolving it via the
                                     # valuesets lookup CTE -- the exact failure mode this schema design
                                     # exists to eliminate, so a correct-by-memorization answer is
                                     # deliberately NOT rewarded at the correct-answer tier. Ranked
                                     # alongside reward_non_executing: not the worst failure (it did execute
                                     # and get the right answer), but not genuine schema/lookup understanding
                                     # either, so it gets no credit for being right.

    # DAPO loss
    eps_low=0.20, eps_high=0.28,   # Clip-Higher -- the paper's own reported values
    ppo_epochs_per_rollout=1,      # gradient passes over each rollout batch before resampling
    train_micro_batch_size=8,      # value used for the published run (Colab G4, 96GB) -- empirically
                                    # stable there; raise cautiously on hardware with more headroom.
    use_torch_compile=False,  # opt-in -- torch.compile via Triton/TorchInductor can speed up training, but this codebase pads to variable shapes every step (rollout completions differ in length, batches aren't padded to a fixed size); full static compilation would recompile on every new shape, which can cost more than it saves. See _maybe_compile below for the dynamic=True mitigation.

    lr=1e-5,             # deliberately far below SFT's 2e-4 -- gentle updates on an already-converged policy
    weight_decay=0.0,
    num_rollout_steps=200,
    early_stopping_patience=3,     # eval-checks (multiples of eval_every_n_steps), not steps --
                                    # stop if best_score hasn't improved in this many consecutive checks.
    save_every_n_steps=2,          # Drive-FUSE sync lag, not a code bug: each checkpoint is
                                    # ~850MB (adapter + AdamW state for 71M params) and Drive's
                                    # background uploader can fall behind every-step writes --
                                    # a hard disconnect then loses whatever hadn't finished
                                    # syncing. Saving less often bounds the redo cost instead.
    eval_every_n_steps=10,
    eval_gen_sample_size=240,      # drives RL training's periodic eval AND the in-corpus test
                                    # comparison's TEST_GEN_SAMPLE (same key). Heldout has its own
                                    # dedicated cap below so changing this doesn't affect that.
    heldout_eval_gen_sample_size=3000,  # separate from eval_gen_sample_size above (which also
                                    # drives RL training's periodic eval and the final in-corpus
                                    # test comparison -- reusing it here would couple heldout's
                                    # scope to those unrelated call sites). At 3000/arm this covers
                                    # the unseen arm in full (2,156 rows) and samples the familiar
                                    # arm (9,300 rows). Full generation coverage of both arms across
                                    # every leg isn't
                                    # computationally tractable even batched (KV-cache scales with
                                    # batch x sequence length, ~1,200-1,500-token schema-dominated
                                    # prompts, ~70GB observed on a 96GB GPU) -- this keeps full
                                    # coverage for the cheap teacher-forced perplexity pass while
                                    # bounding the expensive generation-based metrics to a still-
                                    # large (~10x the original 60) but tractable sample.

    run_heldout_eval=True,         # gen_exec_match is the primary correctness signal on both arms.
                                    # (gen_structure_match predates the lookup redesign: it masks
                                    # code/system literals to avoid conflating SQL skill with code
                                    # recall, but gold SQL now contains no code literals to mask,
                                    # so it collapses to near-exact text match. Kept as a
                                    # diagnostic, not the headline metric. See PAPER.md Sec. 4.)
    seeds=[42, 43, 44],   # one RL run per SFT seed, continuing that seed's own best checkpoint

    # ablation identity -- e.g. 'with_efficiency' vs 'correctness_only' (efficiency_bonus_max=0).
    # Namespaces OUT_BASE below so two ablation arms don't overwrite each other's checkpoints/logs.
    run_name='with_efficiency',
)
print(json.dumps(CFG, indent=2))

assert max(CFG['reward_wrong_result'], CFG['reward_non_executing'],
           CFG['reward_false_abstention'], CFG['reward_missed_abstention'],
           CFG['reward_hardcoded_value']) < 1.0, \
    'every wrong-answer/hardcoded-value reward tier must stay below the correct-answer floor of 1.0'

OUT_BASE = os.path.join(RL_OUTPUTS_ROOT, CFG['run_name'])
os.makedirs(OUT_BASE, exist_ok=True)
print('OUT_BASE =', OUT_BASE)

{
  "model_name": "/content/local/Qwen2.5-Coder-14B-Instruct",
  "quant_type": "nf4",
  "dora_rank": 16,
  "dora_alpha": 32,
  "dora_dropout": 0.05,
  "dora_bias": "none",
  "dora_target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "max_seq_len": 2048,
  "dev_fraction": 0.08,
  "test_fraction": 0.08,
  "split_seed": 20260804,
  "group_size": 4,
  "rollout_groups_per_batch": 8,
  "rollout_prompts_per_batch": 4,
  "max_rollout_attempts": 20,
  "rollout_temperature": 1.0,
  "rollout_top_p": 1.0,
  "max_new_tokens_rollout": 256,
  "efficiency_bonus_max": 0.1,
  "efficiency_min_timing_ms": 0.5,
  "reward_wrong_result": 0.0,
  "reward_non_executing": -0.2,
  "reward_false_abstention": -0.1,
  "reward_missed_abstention": -0.3,
  "eps_low": 0.2,
  "eps_high": 0.28,
  "ppo_epochs_per_rollout": 1,
  "train_micro_batch_size": 4,
  "lr": 1e-05,
  "weight_decay": 0.0,
  "num_rollout_steps": 200,
  "early_stopping_patience

## DAPO core: loss, advantage, dynamic-sampling filter

Four functions, reused as-is:
- `compute_dapo_loss` -- Clip-Higher PPO loss (asymmetric `eps_low`/`eps_high`), token-level
  normalized (sum over all response tokens in the batch, divided by the total token count).
- `gather_token_logprobs` -- per-token log-probability of the actual next token, shifted for
  causal LM (`logits[:, :-1, :]` against `input_ids[:, 1:]`). Its second parameter is the full
  `input_ids` tensor -- called `responses_mask` in the signature, but used as `input_ids` inside
  the function; called accordingly below, not by its parameter name.
- `group_advantages` -- within-group reward normalization: `(r - group_mean) / group_std`, the
  GRPO-style advantage estimate that replaces a learned value/critic network.
- `keep_groups` -- Dynamic Sampling: `True` for a group only if its rewards have nonzero
  variance (not all-correct or all-wrong), since a zero-variance group's advantage is exactly
  zero for every sample in it and contributes no gradient signal.

In [ ]:
def compute_dapo_loss(logprobs_new, logprobs_old, advantages, mask,
                      eps_low=0.2, eps_high=0.28):
    # logprobs_*: [B, T-1]   advantages: [B]   mask: [B, T-1]
    ratio = torch.exp(logprobs_new - logprobs_old)      # [B, T-1]
    adv   = advantages.unsqueeze(-1)                    # [B, 1]  broadcasts over tokens

    unclipped = ratio * adv                             # [B, T-1]
    clipped   = torch.clamp(ratio, 1 - eps_low, 1 + eps_high) * adv   # [B, T-1]
    per_token = torch.min(unclipped, clipped)           # [B, T-1]

    # DAPO token-level normalization: global sum / total tokens
    loss = -(per_token * mask).sum() / mask.sum().clamp(min=1.0)
    return loss                                         # scalar


def gather_token_logprobs( logits, responses_mask):
    # logits: [B, T, V]   input_ids: [B, T]
    logits=logits[:,:-1,:]  # [B, T-1, V]
    input_ids=responses_mask[:,1:]  # [B, T-1]
    # 1. log_softmax over vocab
    log_probs=F.log_softmax(logits,dim=-1)  # [B, T, V]
    # 2. gather the actual next-token log-prob (mind the t -> t+1 shift)
    log_probs=torch.gather(log_probs,dim=-1,index=input_ids.unsqueeze(-1)).squeeze(-1)
    # return: [B, T-1]
    return log_probs

def group_advantages( rewards,group_size):
    # rewards: [B]  one scalar per response (B = num_prompts * group_size)
    # normalize within each group: (r - mean) / std
    # return: [B]
    rewards = rewards.view(-1, group_size)  # [num_prompts, group_size]
    mean=rewards.mean(dim=-1,keepdim=True)
    std=rewards.std(dim=-1,keepdim=True)
    adv= (rewards - mean) / (std+1e-8)  # [B]
    return adv.view(-1)  # [B]

def keep_groups(rewards, group_size, threshold=1e-6):
    # rewards: [B]  flat, B = num_prompts * group_size
    rewards=rewards.view(-1,group_size)  # [num_prompts, group_size]
    # return a BOOLEAN mask [num_prompts] -- True = keep (std > 0), False = drop
    mask = rewards.std(dim=-1) > threshold  # [num_prompts]
    return mask

## Model wrapper

Same `FHIRSQLLLM` as the SFT notebook -- used here only for `_load_tokenizer()` and
`_load_base_model()`. The RL adapter itself is not built by `_get_dora()`; it's loaded directly
from a seed's SFT `best/` checkpoint (see the Trainer section), so RL continues the exact adapter
weights SFT produced rather than starting a fresh one.

In [ ]:
def _pick_attn_implementation():
    '''Flash Attention 2 needs the flash-attn package AND Ampere+ (compute capability >= 8.0) --
    neither is guaranteed on every GPU this notebook might run on, so this probes rather than
    assumes, falling back to
    PyTorch's own SDPA (already a fused, efficient attention kernel, just without FA2's specific
    memory/speed profile) instead of failing outright if FA2 isn't available or isn't supported.'''
    if not torch.cuda.is_available():
        return 'sdpa'
    try:
        import flash_attn  # noqa: F401
    except ImportError:
        return 'sdpa'
    major, _minor = torch.cuda.get_device_capability()
    if major < 8:
        return 'sdpa'
    return 'flash_attention_2'


class FHIRSQLLLM:
    '''Base model + DoRA adapter wrapper.'''

    def __init__(self, cfg):
        self.model_name = cfg.get('model_name', 'Qwen/Qwen2.5-Coder-14B-Instruct')
        self.quant_type = cfg.get('quant_type', 'nf4')
        self.dtype = torch.bfloat16
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.dora_rank = cfg.get('dora_rank', 16)
        self.dora_alpha = cfg.get('dora_alpha', 32)
        self.dora_dropout = cfg.get('dora_dropout', 0.05)
        self.dora_bias = cfg.get('dora_bias', 'none')
        self.dora_target_modules = cfg.get(
            'dora_target_modules',
            ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        )

    def _load_tokenizer(self):
        tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        tokenizer.padding_side = 'right'
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        return tokenizer

    def _load_base_model(self):
        attn_impl = _pick_attn_implementation()
        model = AutoModelForCausalLM.from_pretrained(
            self.model_name, dtype=self.dtype, quantization_config=self._get_quantization_config(),
            attn_implementation=attn_impl,
        )
        print(f'attn_implementation = {attn_impl}')
        model.config.use_cache = False
        return prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    def _get_quantization_config(self):
        return BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=self.dtype,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type=self.quant_type,
        )

    def _get_dora(self):
        return LoraConfig(
            r=self.dora_rank, lora_alpha=self.dora_alpha, bias=self.dora_bias,
            lora_dropout=self.dora_dropout, use_dora=True,
            target_modules=self.dora_target_modules, task_type=TaskType.CAUSAL_LM,
        )

    def inspect_trainable_parameters(self, model):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        names = [n for n, p in model.named_parameters() if p.requires_grad]
        print(f'total params: {total:,} | trainable: {trainable:,} ({100*trainable/total:.3f}%)')
        assert trainable > 0
        assert any('lora_A' in n for n in names)
        assert any('lora_B' in n for n in names)
        assert any('lora_magnitude_vector' in n for n in names)

## Data loading, prompt format, and reward

Same schema-DDL extraction, system prompt, and grouped/stratified train-dev-test split as SFT
(identical `split_seed`, so this reconstructs the exact same three-way partition) -- RL only
rolls out prompts from `train_rows`, and only ever evaluates checkpoints against `dev_rows`
(periodically, for best-checkpoint selection) and `test_rows` (once, at the end), never training
on either.

The reward is correctness-gated with a small execution-efficiency bonus on top: 1.0 if the gold
is `UNANSWERABLE` and the model correctly abstains, or if the gold is SQL and the model's SQL
executes against `train.duckdb` with the same result set (order-insensitive) as the gold query
-- plus up to `efficiency_bonus_max` extra if the correct query also ran faster than gold's own
execution (both timed fresh, in the same call, for a fair contemporaneous comparison; timings
are floored at `efficiency_min_timing_ms` so measurement noise on sub-millisecond queries can't
produce a wild speedup ratio). 0.0 for anything wrong -- non-executing SQL, wrong result, false
abstention, missed abstention -- with no efficiency bonus possible, so a faster wrong query can
never outscore a correct one. This also means a group where every sample is correct (previously
reward=1.0 for all four, exactly zero variance, dropped by Dynamic Sampling as uninformative)
can now carry a small efficiency-driven variance and get *kept* -- which is the point: it lets
the model learn to prefer the more efficient of several equally-correct queries, rather than
discarding that comparison as noise.

In [ ]:
ABSTENTION_TOKEN = 'UNANSWERABLE'

SYSTEM_PROMPT_TEMPLATE = '''You are a clinical data analyst who translates natural-language hospital \
questions into DuckDB SQL, run against the schema below, via an explicit query plan first.

Output exactly two parts, in this order, and nothing else:
1. A JSON object describing the query plan: which clinical concepts the question refers to (and \
whether each needs a terminology lookup against the schema's `valuesets` table), what \
additional tables must be joined and why, what filters apply, and what the final aggregation \
computes.
2. The compiled SQL statement for that plan, in a fenced code block:
```sql
<the SQL statement>
```

If the question cannot be answered from this schema -- the data it needs genuinely doesn't \
exist here -- the plan should be {{"abstain": true}}, and the fenced SQL block should contain \
exactly the single word: {abstention_token}

Do not guess or approximate an answer using unrelated columns when the real field is absent.

Schema:
{schema_ddl}'''


def load_schema_ddl(schema_path):
    text = open(schema_path, encoding='utf-8').read()
    idx = text.index('CREATE TABLE')
    return text[idx:].strip()


def build_messages(row, schema_ddl):
    system = SYSTEM_PROMPT_TEMPLATE.format(abstention_token=ABSTENTION_TOKEN, schema_ddl=schema_ddl)
    return [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': row['question']},
        {'role': 'assistant', 'content': row['target']},
    ]


def chat_template_ids(tokenizer, messages, add_generation_prompt):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)
    return tokenizer(text, add_special_tokens=False)['input_ids']


_OPERATIONAL_IDS = {
    'doctor_top5_by_encounter_type', 'doctor_top5_prescribers', 'doctor_top5_by_condition_diagnosed',
    'bed_occupancy_at_date', 'bed_occupancy_by_month', 'avg_length_of_stay',
    'current_inpatient_census_by_month',
}
_QUALITY_KPI_IDS = {
    'condition_by_race', 'condition_by_ethnicity', 'physician_monthly_case_volume',
    'readmission_rate_30day', 'mortality_review_monthly', 'fall_risk_screening_monthly',
    'patients_by_race', 'patients_by_ethnicity', 'patients_by_state',
    'imaging_volume_by_month', 'imaging_volume_by_modality',
}


def archetype_category(archetype_id):
    if archetype_id.startswith('unans_'):
        return 'unanswerable'
    if archetype_id.startswith('reg_'):
        return 'regulatory'
    if archetype_id in _OPERATIONAL_IDS:
        return 'operational'
    if archetype_id in _QUALITY_KPI_IDS:
        return 'quality_kpi_demographics'
    return 'main'


def instance_key(row):
    return f"{row['archetype_id']}|{row.get('concept_display')}|{row['gold_sql']}"


def grouped_split(rows, dev_fraction, test_fraction, seed):
    groups = {}
    stratum_of = {}
    for r in rows:
        k = instance_key(r)
        groups.setdefault(k, []).append(r)
        stratum_of[k] = (r['tier'], archetype_category(r['archetype_id']))
    by_stratum = {}
    for k, stratum in stratum_of.items():
        by_stratum.setdefault(stratum, []).append(k)
    rng = random.Random(seed)
    dev_keys, test_keys = set(), set()
    for stratum in sorted(by_stratum.keys()):
        keys = sorted(by_stratum[stratum])
        rng.shuffle(keys)
        n = len(keys)
        if n < 3:
            continue
        n_dev = max(1, round(n * dev_fraction))
        n_test = max(1, round(n * test_fraction))
        while n_dev + n_test > n - 1:
            if n_dev >= n_test:
                n_dev -= 1
            else:
                n_test -= 1
        dev_keys.update(keys[:n_dev])
        test_keys.update(keys[n_dev:n_dev + n_test])
    train_rows, dev_rows, test_rows = [], [], []
    for k, rs in groups.items():
        if k in dev_keys:
            dev_rows.extend(rs)
        elif k in test_keys:
            test_rows.extend(rs)
        else:
            train_rows.extend(rs)
    return train_rows, dev_rows, test_rows


rows = [json.loads(l) for l in open(LOCAL_JSONL, encoding='utf-8')]
schema_ddl = load_schema_ddl(LOCAL_SCHEMA)
train_rows, dev_rows, test_rows = grouped_split(rows, CFG['dev_fraction'], CFG['test_fraction'], CFG['split_seed'])
print(f'rows: {len(rows)} | train: {len(train_rows)} | dev: {len(dev_rows)} | test: {len(test_rows)}')

GEN_EVAL_SAMPLE = random.Random(CFG['split_seed']).sample(
    dev_rows, min(CFG['eval_gen_sample_size'], len(dev_rows))
)
TEST_GEN_SAMPLE = random.Random(CFG['split_seed']).sample(
    test_rows, min(CFG['eval_gen_sample_size'], len(test_rows))
)
print(f'fixed eval samples: dev={len(GEN_EVAL_SAMPLE)} rows, test={len(TEST_GEN_SAMPLE)} rows')


def _round_val(v, ndigits=2):
    # Float-tolerant comparison: DuckDB's parallel aggregation reorders
    # floating-point summation non-associatively across threads, so byte-identical SQL run twice
    # can return values differing at the 10th-15th decimal place -- confirmed directly by
    # repeated execution of real gold SQL (61/80 AVG(valueQuantity) queries differed across 5
    # runs before this fix). Gold SQL now rounds AVG(valueQuantity) to 4 decimals at the source
    # (see archetypes.py), which fixes ~95% of cases, but GROUP BY-year aggregates can still
    # occasionally straddle a rounding boundary (e.g. 4.1260 vs 4.1261) since that's a genuine
    # boundary-flip, not raw jitter, and no amount of added SQL-level decimal precision
    # eliminates a boundary problem -- only a coarser comparison tolerance does. Rounding to 2
    # decimals here gives 100x margin over the observed ~0.0001 residual, applied uniformly (not
    # just to the known-affected archetypes) so any other float-producing SQL is covered too.
    if isinstance(v, float):
        return round(v, ndigits)
    return v


def _exec_rows(con, sql):
    # Plain sorted() crashes ('<' not supported between float and NoneType) the moment a result
    # column mixes NULL and non-NULL values across rows -- e.g. AVG() returning NULL for one
    # group but a real number for another. Both callers of this function catch exceptions and
    # score the query as wrong (0.0 / no match) on any error, so this was silently mis-scoring
    # otherwise-correct queries rather than crashing loudly. Sorting on (is_none, value) per
    # column keeps None grouped and orderable without ever comparing it to a real value.
    rows = [tuple(_round_val(v) for v in row) for row in con.execute(sql).fetchall()]
    return sorted(rows, key=lambda row: tuple((v is None, v) for v in row))


_CODE_LITERAL_RE = re.compile(r"(code\s*=\s*)'[^']*'", re.IGNORECASE)
_SYSTEM_LITERAL_RE = re.compile(r"(system\s*=\s*)'[^']*'", re.IGNORECASE)


def _normalize_structure(sql):
    """Strips the specific code/system LITERAL out of a WHERE clause, keeping everything else --
    used for gen_structure_match: 'same archetype/table/query shape, regardless of
    which literal code was used.' Deliberately narrower than a real SQL parser (doesn't handle
    every possible formatting variant), but the model's own output closely mirrors gold's
    formatting after SFT, so exact-modulo-code-value comparison is precise enough in practice.
    Not used for training reward (compute_reward) -- eval-only, since 'is this fine to report as
    correct on the unseen arm' and 'is this fine to reward during RL training' are different
    questions."""
    s = _CODE_LITERAL_RE.sub(r"\1'<CODE>'", sql)
    s = _SYSTEM_LITERAL_RE.sub(r"\1'<SYS>'", s)
    return ' '.join(s.split())


_SQL_FENCE_RE = re.compile(r"```sql\s*\n(.*?)\n```", re.IGNORECASE | re.DOTALL)


def extract_sql_from_completion(text):
    '''Pulls the SQL out of a `plan JSON` + fenced-```sql-block completion.
    Takes the LAST matching fence, not the first -- if the model includes an earlier example or
    stray fence while "thinking," the final one is the one meant as the actual answer, same
    convention as extracting a final boxed answer in reasoning-then-answer RL setups generally.
    Returns None (not a crash, not an empty string) when no fence is found at all, so callers can
    treat "no parseable completion" as its own explicit case rather than accidentally executing
    an empty SQL string.'''
    matches = _SQL_FENCE_RE.findall(text)
    if not matches:
        return None
    return matches[-1].strip()


_HARDCODED_TERMINOLOGY_RE = re.compile(
    r"\b(?:code|system|type_code|type_system|vaccineCode|vaccineCode_system)\s*=\s*'[^']*'",
    re.IGNORECASE,
)


def has_hardcoded_terminology(sql):
    '''True if `sql` compares a terminology code/system column directly to a quoted literal,
    instead of via the lookup-CTE join pattern (`WHERE <table>.code = resolved.code`, an
    identifier comparison, never matches this). This is a correct detector precisely BECAUSE
    every legitimate lookup CTE in this design only ever filters on `table_name`/`display`
    (never `code`/`system` themselves) -- a match here can only mean the model bypassed the
    lookup mechanism entirely, primary OR companion concept, not a false positive against the
    CTE's own internals. Deliberately narrower than a real SQL parser (column-name-based, not
    schema-aware), consistent with _normalize_structure's existing precedent in this file --
    non-terminology literal filters (status/criticality/class_code/gender/dates) use different
    column names and are never flagged.'''
    return bool(_HARDCODED_TERMINOLOGY_RE.search(sql))


def compute_reward(pred_text, gold_sql, duckdb_con, cfg):
    '''Wrong answers are tiered, not a flat 0.0 -- see CFG for the ranking rationale.
    A group of completions that fail in different ways now has real reward variance instead of
    all collapsing to the same value, so Dynamic Sampling can keep and learn from groups that
    would previously have looked uniform and been dropped.

    pred_text is a full plan+SQL completion, not raw SQL -- extract_sql_from_completion pulls the
    SQL out first. A missing/unparseable fence scores reward_non_executing (same tier as SQL that
    fails to execute -- both are "the model didn't produce usable SQL"), which the caller can
    additionally tag as a distinct 'malformed_completion' failure type for diagnostics without
    changing the reward itself. The plan JSON is never separately rewarded -- only the compiled
    SQL's correctness + efficiency.'''
    gold = gold_sql.strip()
    is_abst_gold = gold == ABSTENTION_TOKEN
    pred_sql = extract_sql_from_completion(pred_text)
    if pred_sql is None:
        return cfg['reward_non_executing']
    pred = pred_sql.strip()
    is_abst_pred = pred == ABSTENTION_TOKEN
    if is_abst_gold:
        return 1.0 if is_abst_pred else cfg['reward_missed_abstention']
    if is_abst_pred:
        return cfg['reward_false_abstention']
    # Untimed warm-up pass -- also serves as the correctness check. Whichever query is timed
    # SECOND below would otherwise inherit whatever pages this pass pulled into DuckDB's buffer
    # pool, biasing the comparison; running both once, untimed, first equalizes that.
    try:
        pred_rows = _exec_rows(duckdb_con, pred)
    except Exception:
        return cfg['reward_non_executing']
    gold_rows = _exec_rows(duckdb_con, gold)
    if pred_rows != gold_rows:
        return cfg['reward_wrong_result']

    # Correct result, but did it get there by hardcoding a code instead of using the lookup
    # CTE? Checked before the efficiency bonus deliberately -- a hardcoded answer earns no
    # efficiency credit either, only ever the flat penalty tier.
    if has_hardcoded_terminology(pred):
        return cfg['reward_hardcoded_value']

    # Timed pass: both queries are now warm, so this measures the queries, not cold-cache
    # penalties. Order randomized so neither systematically inherits the other's warm state
    # from THIS pass either.
    order = ['pred', 'gold'] if random.random() < 0.5 else ['gold', 'pred']
    timings = {}
    for which in order:
        sql = pred if which == 'pred' else gold
        t0 = time.perf_counter()
        _exec_rows(duckdb_con, sql)
        timings[which] = time.perf_counter() - t0

    floor_s = cfg['efficiency_min_timing_ms'] / 1000
    speedup = max(timings['gold'], floor_s) / max(timings['pred'], floor_s)
    bonus = cfg['efficiency_bonus_max'] * max(0.0, min(1.0, speedup - 1.0))
    return 1.0 + bonus


duckdb_con = duckdb.connect(LOCAL_DUCKDB, read_only=True)
print('execution-eval DB connected:', LOCAL_DUCKDB)

rows: 11844 | train: 9956 | dev: 944 | test: 944
fixed eval samples: dev=944 rows, test=944 rows
execution-eval DB connected: /content/local/train.duckdb


## Teacher-forced eval machinery (reused from SFT)

`SFTDataset`/`collate_fn`/`evaluate_loss_and_token_acc`/`evaluate_generation` are the same
functions used in `sft_train.ipynb`, copied here so RL's final report is directly comparable in
shape to `sft_outputs/seed_<seed>/test_eval.json` -- same fields, same computation. Teacher-forced
loss/perplexity on the gold SQL isn't what RL optimizes (reward is), but tracking it is a useful
check that RL hasn't destroyed the model's ability to reproduce SFT-style output, and it keeps the
final SFT-vs-RL comparison table apples-to-apples.

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, rows, tokenizer, schema_ddl, max_seq_len):
        self.rows = rows
        self.tokenizer = tokenizer
        self.schema_ddl = schema_ddl
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        messages = build_messages(row, self.schema_ddl)
        prompt_ids = chat_template_ids(self.tokenizer, messages[:-1], add_generation_prompt=True)
        full_ids = chat_template_ids(self.tokenizer, messages, add_generation_prompt=False)
        full_ids = full_ids[: self.max_seq_len]
        prompt_len = min(len(prompt_ids), len(full_ids))
        assert prompt_len < len(full_ids), (
            f'row {i} (archetype={row["archetype_id"]!r}): truncation to max_seq_len='
            f'{self.max_seq_len} left no room for the answer.'
        )
        labels = [-100] * prompt_len + full_ids[prompt_len:]
        return {'input_ids': full_ids, 'labels': labels}


def collate_fn(batch, pad_token_id):
    max_len = max(len(b['input_ids']) for b in batch)
    input_ids, labels, attn = [], [], []
    for b in batch:
        pad = max_len - len(b['input_ids'])
        input_ids.append(b['input_ids'] + [pad_token_id] * pad)
        labels.append(b['labels'] + [-100] * pad)
        attn.append([1] * len(b['input_ids']) + [0] * pad)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
        'attention_mask': torch.tensor(attn, dtype=torch.long),
    }


def _worker_init(_wid):
    try:
        torch.set_num_threads(1)
    except Exception:
        pass


def _loader_kwargs(cfg):
    nw = cfg.get('num_workers', 2)
    kw = dict(num_workers=nw, pin_memory=(nw > 0 and torch.cuda.is_available()))
    if nw > 0:
        kw.update(persistent_workers=True, worker_init_fn=_worker_init, prefetch_factor=2)
    return kw


def make_loader(rows_, tokenizer, schema_ddl_, cfg, shuffle):
    # batch_size is hardcoded to 1: fine for RL training's own per-sample
    # rollout path, which never calls this, but this loader is eval-only (teacher-forced
    # loss/perplexity/token-acc, always under torch.no_grad) and collate_fn already implements
    # correct padding + attention_mask + label masking for any batch size -- no other code needs
    # to change. Reuses train_micro_batch_size as an already-probed-safe default; forward-only
    # (no backward, no optimizer state) could likely go higher, but this avoids a new guess.
    ds = SFTDataset(rows_, tokenizer, schema_ddl_, cfg['max_seq_len'])
    collate = functools.partial(collate_fn, pad_token_id=tokenizer.pad_token_id)
    bs = cfg.get('eval_loader_batch_size', cfg.get('train_micro_batch_size', 1))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=shuffle,
                       collate_fn=collate, **_loader_kwargs(cfg))


@torch.no_grad()
def evaluate_loss_and_token_acc(model, loader, amp_ctx_fn):
    model.eval()
    total_loss, total_tokens, correct_tokens, n_batches = 0.0, 0, 0, 0
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        with amp_ctx_fn():
            out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
        total_loss += out.loss.item(); n_batches += 1
        preds = out.logits[:, :-1, :].argmax(-1)
        tgt = labels[:, 1:]
        mask = tgt != -100
        correct_tokens += (preds[mask] == tgt[mask]).sum().item()
        total_tokens += mask.sum().item()
    mean_loss = total_loss / max(1, n_batches)
    return {'loss': mean_loss, 'perplexity': math.exp(mean_loss),
            'token_acc': correct_tokens / max(1, total_tokens)}


@torch.no_grad()
def evaluate_generation(model, tokenizer, sample_rows, schema_ddl_, cfg, duckdb_con_, failure_log_path=None):
    '''gen_efficiency_speedup: median (gold execution time / predicted execution time) among rows
    scored exec-correct -- NaN if none were correct, since there's nothing to measure efficiency
    over. Measured the same cache-fair way as the training reward (compute_reward): an untimed
    warm-up pass on both queries first (equalizes DuckDB buffer-pool state), then a timed second
    execution of each with randomized order, so this metric and the reward it's meant to explain
    use the same methodology, not two different ones that could disagree for unrelated reasons.

    gen_structure_match: does predicted SQL match gold with the literal code/system VALUE stripped
    out (see _normalize_structure)? Same archetype/table/query shape counts as a match even if the
    specific code is wrong -- a separate notion of correctness from gen_exact_match/gen_exec_match,
    which conflate 'wrote correct SQL' with 'recalled the right clinical code from memory' for
    concept-parameterized archetypes. NOTE: that rationale predates the lookup-CTE redesign -- gold
    SQL no longer contains code literals, so the masking is inert and this metric collapses to
    near-exact text match. gen_exec_match is the primary correctness signal; this is a diagnostic.

    Batched: processes eval_gen_batch_size prompts per generate() call instead of
    one at a time. Unbatched was fine at the original small eval_gen_sample_size (60 rows) but
    became a multi-hour bottleneck once the heldout eval sample was raised to cover full arms
    (thousands of rows) -- same fixed per-call overhead problem already fixed for training
    rollout, just never applied to evaluation. No new batch-size probe needed: eval is greedy
    (do_sample=False, one sequence per prompt, no group_size replication), so it's strictly
    lighter per call than the rollout probe already validated at rollout_prompts_per_batch x
    group_size concurrent sequences -- reusing that number here is a safe, already-proven bound,
    not a new guess.'''
    model.eval()
    prev_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    batch_size = cfg.get('eval_gen_batch_size', cfg['rollout_prompts_per_batch'] * cfg['group_size'])

    n_exact = 0
    n_structure_match = 0
    n_hardcoded = 0   # of the exec-correct rows, how many got there by
                      # hardcoding a code instead of using the lookup CTE -- see
                      # has_hardcoded_terminology / reward_hardcoded_value. Purely a
                      # diagnostic here (not reward-bearing in eval), tracked so training
                      # progress on this specific failure mode is visible over time.
    abst_tp = abst_fp = abst_fn = abst_tn = 0
    exec_match = exec_total = 0
    speedups = []
    # failure_log_path: appends one JSON line per failing row as it's found (not
    # buffered to the end), so an interrupted run still leaves whatever it processed on disk.
    # Same four-way taxonomy as the RL reward tiers (wrong_result/non_executing/false_abstention/
    # missed_abstention) so failure logs and training reward penalties line up directly. Off by
    # default -- this function also runs on every RL training-step eval, which doesn't want a
    # growing failure log.
    failure_file = open(failure_log_path, 'a', encoding='utf-8') if failure_log_path else None

    def _log_failure(row, pred, gold, failure_type, exec_error=None):
        if failure_file is None:
            return
        rec = {
            'archetype_id': row.get('archetype_id'), 'tier': row.get('tier'),
            'question': row.get('question'), 'gold_sql': gold, 'pred_sql': pred,
            'failure_type': failure_type, 'exec_error': exec_error,
        }
        failure_file.write(json.dumps(rec) + '\n')
        failure_file.flush()

    for start in range(0, len(sample_rows), batch_size):
        batch_rows = sample_rows[start:start + batch_size]
        prompt_texts = []
        for r in batch_rows:
            messages = build_messages(r, schema_ddl_)[:-1]
            prompt_texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

        enc = tokenizer(prompt_texts, return_tensors='pt', padding=True, truncation=True,
                         max_length=cfg['max_seq_len'], add_special_tokens=False).to(DEVICE)
        padded_prompt_len = enc['input_ids'].shape[1]

        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=cfg.get('max_new_tokens_eval', 2048), do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)

        for i, r in enumerate(batch_rows):
            # skip_special_tokens strips eos/pad regardless of where a shorter sequence in this
            # batch finished early -- same implicit boundary handling the original unbatched code
            # already relied on, no explicit EOS-position search needed here (unlike training
            # rollout, which needs the raw token ids for old-logprob computation afterward; this
            # is eval-only, we just need the decoded text).
            completion = tokenizer.decode(gen[i][padded_prompt_len:], skip_special_tokens=True).strip()
            pred_sql = extract_sql_from_completion(completion)
            gold = r['gold_sql'].strip()
            if pred_sql is None:
                # No parseable ```sql fence at all -- distinct from "SQL that fails to execute,"
                # logged separately for diagnostics even though it's scored the same downstream.
                _log_failure(r, completion, gold, 'malformed_completion')
                pred_sql = ''   # falls through as neither abstention nor any gold match below
            pred = pred_sql.strip()
            is_abst_gold = gold == ABSTENTION_TOKEN
            is_abst_pred = pred == ABSTENTION_TOKEN
            if is_abst_gold and is_abst_pred: abst_tp += 1
            elif is_abst_gold and not is_abst_pred:
                abst_fn += 1
                _log_failure(r, pred, gold, 'missed_abstention')
            elif not is_abst_gold and is_abst_pred:
                abst_fp += 1
                _log_failure(r, pred, gold, 'false_abstention')
            else: abst_tn += 1
            if not is_abst_gold:
                if pred == gold:
                    n_exact += 1
                if not is_abst_pred and pred and _normalize_structure(pred) == _normalize_structure(gold):
                    n_structure_match += 1
                if duckdb_con_ is not None and not is_abst_pred:
                    # pred=='' (malformed completion, already logged above) falls through to the
                    # exec attempt below rather than being skipped -- _exec_rows('') raises, which
                    # the existing try/except turns into a correctly-counted exec_total failure
                    # (not silently excluded from the denominator, which would inflate gen_exec_match).
                    exec_total += 1
                    exec_error = None
                    try:
                        is_correct = _exec_rows(duckdb_con_, pred) == _exec_rows(duckdb_con_, gold)
                    except Exception as e:
                        is_correct = False
                        exec_error = str(e)
                    if is_correct:
                        exec_match += 1
                        if has_hardcoded_terminology(pred):
                            n_hardcoded += 1
                        try:
                            _exec_rows(duckdb_con_, pred); _exec_rows(duckdb_con_, gold)   # untimed warm-up, both
                            order = ['pred', 'gold'] if random.random() < 0.5 else ['gold', 'pred']
                            timings = {}
                            for which in order:
                                sql = pred if which == 'pred' else gold
                                t0 = time.perf_counter()
                                _exec_rows(duckdb_con_, sql)
                                timings[which] = time.perf_counter() - t0
                            floor_s = cfg.get('efficiency_min_timing_ms', 0.5) / 1000
                            speedups.append(max(timings['gold'], floor_s) / max(timings['pred'], floor_s))
                        except Exception:
                            pass   # correctness already scored above; a timing hiccup shouldn't drop the row
                    else:
                        _log_failure(r, pred, gold, 'non_executing' if exec_error else 'wrong_result', exec_error)

    tokenizer.padding_side = prev_padding_side
    if failure_file is not None:
        failure_file.close()
    n_answerable = sum(1 for r in sample_rows if r['gold_sql'] != ABSTENTION_TOKEN)
    return {
        'gen_exact_match': n_exact / max(1, n_answerable),
        'gen_structure_match': n_structure_match / max(1, n_answerable),
        'gen_exec_match': (exec_match / exec_total) if exec_total else float('nan'),
        'gen_abstention_precision': abst_tp / max(1, abst_tp + abst_fp),
        'gen_abstention_recall': abst_tp / max(1, abst_tp + abst_fn),
        'gen_efficiency_speedup': statistics.median(speedups) if speedups else float('nan'),
        'gen_hardcoded_rate': (n_hardcoded / exec_match) if exec_match else float('nan'),
    }

## Logging and checkpointing (reused from SFT)

In [ ]:
class TeeFile:
    def __init__(self, *targets):
        self.files, self._opened = [], []
        for t in targets:
            if isinstance(t, str):
                f = open(t, 'a', buffering=1); self.files.append(f); self._opened.append(f)
            else:
                self.files.append(t)
    def write(self, data):
        for f in self.files:
            try: f.write(data); f.flush()
            except Exception: pass
    def flush(self):
        for f in self.files:
            try: f.flush()
            except Exception: pass
    def close(self):
        for f in self._opened:
            try: f.close()
            except Exception: pass


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def _maybe_compile(model, cfg):
    '''torch.compile (Triton/TorchInductor) -- opt-in via CFG['use_torch_compile'], default
    False. This codebase deliberately handles variable-shape batches every step (rollout
    completions differ in length, no fixed-length padding), so a full static compile would
    recompile on every new shape, which can cost more wall-clock than it saves. dynamic=True
    compiles a shape-flexible graph instead, trading some peak speedup for robustness to that
    variability -- still experimental for a 4-bit-quantized + PEFT model (bitsandbytes' custom
    autograd ops are a known source of graph breaks that can silently reduce or eliminate the
    speedup), so test on a handful of real steps before trusting it for a full run.'''
    if not cfg.get('use_torch_compile', False):
        return model
    if not (torch.cuda.is_available() and hasattr(torch, 'compile')):
        return model
    return torch.compile(model, dynamic=True)


def _atomic_save_adapter(model, path, retries=6, delay=2.0):
    '''Never has a window where `path` holds neither a complete old nor complete new checkpoint --
    unlike a naive rmtree(path)-then-replace(tmp, path), which briefly deletes path outright before
    the replacement lands. Writes to tmp, demotes the current path to path+'.prev' (a fast metadata
    rename, not a copy -- safe even for a multi-hundred-MB directory), promotes tmp to path, THEN
    drops .prev. If a crash lands between the demote and promote (a few-millisecond window instead
    of however long shutil.rmtree+os.replace used to take), the caller can still resume from
    path+'.prev' -- see RLTrainer._build_model_and_tokenizer.'''
    tmp = path + f'.tmp{os.getpid()}'
    prev = path + '.prev'
    for attempt in range(retries):
        try:
            if os.path.exists(tmp):
                shutil.rmtree(tmp)
            model.save_pretrained(tmp)
            if os.path.exists(prev):
                shutil.rmtree(prev)
            if os.path.exists(path):
                os.replace(path, prev)
            os.replace(tmp, path)
            if os.path.exists(prev):
                shutil.rmtree(prev)
            return
        except (RuntimeError, OSError, PermissionError) as e:
            if attempt == retries - 1:
                print(f'  [warn] adapter save failed after {retries} tries ({e}); continuing')
                return
            time.sleep(delay)


def _atomic_torch_save(obj, path, retries=6, delay=1.5):
    d = os.path.dirname(path); base = os.path.basename(path)
    for attempt in range(retries):
        tmp = os.path.join(d, f'.{base}.tmp{os.getpid()}')
        try:
            torch.save(obj, tmp); os.replace(tmp, path); return
        except (RuntimeError, OSError, PermissionError) as e:
            try:
                if os.path.exists(tmp): os.remove(tmp)
            except Exception: pass
            if attempt == retries - 1:
                print(f'  [warn] state save failed after {retries} tries ({e}); continuing'); return
            time.sleep(delay)

## Rollout and Dynamic Sampling

`CyclicPromptSampler` replaces plain `random.choice` (with replacement) for drawing prompts. With replacement, coverage of the ~9,956-row train pool over a fixed step budget is probabilistic and incomplete (measured: ~15% of the pool touched even once over 200 steps, ~73% over 1200) -- the cyclic sampler shuffles the full pool once, draws without replacement until exhausted, then reshuffles for a fresh cycle, guaranteeing every training instance is sampled at least once per `len(prompt_pool)` draws instead of leaving it to chance.

`sample_group_batch` generates `group_size` completions for several DIFFERENT prompts in one
batched `model.generate()` call (left-padded, stripped back to padding-free sequences immediately
after), rather than one `generate()` call per prompt -- reaching `rollout_groups_per_batch` kept
groups needs ~10-12 attempts/step (Dynamic Sampling drops some), so this amortizes NF4 dequant +
kernel-launch overhead across `rollout_prompts_per_batch x group_size` sequences per call instead
of paying it 10-12 times over. `collect_rollout_batch` samples prompts in chunks of
`rollout_prompts_per_batch` until `rollout_groups_per_batch` groups are kept or
`max_rollout_attempts` total prompts have been tried (same semantics as before -- a safety cap on
total prompts sampled, just now sampled in batches instead of one at a time).

In [ ]:
class CyclicPromptSampler:
    '''Samples without replacement from prompt_pool, reshuffling into a fresh cycle once
    exhausted -- guarantees every prompt is drawn at least once every len(prompt_pool) calls,
    unlike random.choice (with replacement), which can leave a large fraction of the pool
    completely unseen over a run's step budget. State is checkpointed (see RLTrainer._save /
    _maybe_resume_training_state) so a resume continues the same cycle rather than restarting it.'''
    def __init__(self, prompt_pool, seed):
        self.prompt_pool = prompt_pool
        self.rng = random.Random(seed)
        self._order = []
        self._pos = 0
        self.cycle_count = 0
        self.on_reshuffle = None   # optional callback(cycle_count) -- RLTrainer wires this to _log

    def _reshuffle(self):
        self._order = list(range(len(self.prompt_pool)))
        self.rng.shuffle(self._order)
        self._pos = 0
        self.cycle_count += 1
        if self.on_reshuffle is not None:
            self.on_reshuffle(self.cycle_count)

    def sample(self, n):
        out = []
        for _ in range(n):
            if self._pos >= len(self._order):
                self._reshuffle()
            out.append(self.prompt_pool[self._order[self._pos]])
            self._pos += 1
        return out

    def state_dict(self):
        return {'order': self._order, 'pos': self._pos, 'cycle_count': self.cycle_count}

    def load_state_dict(self, state):
        self._order = state['order']; self._pos = state['pos']; self.cycle_count = state['cycle_count']


@torch.no_grad()
def sample_group_batch(model, tokenizer, rows, schema_ddl_, cfg, duckdb_con_):
    '''Generates group_size completions for EACH of len(rows) different prompts in one batched,
    left-padded model.generate() call, instead of one generate() call per prompt -- amortizes
    NF4 dequant + kernel-launch overhead across len(rows)*group_size sequences instead of paying
    it once per prompt (the dominant cost: reaching rollout_groups_per_batch
    kept groups needs ~10-12 attempts/step, each previously a separate small generate() call).
    Returns a list of len(rows) group-dicts (samples/rewards/row), one per input row.'''
    model.eval()
    group_size = cfg['group_size']
    n_prompts = len(rows)

    prompt_texts, real_prompt_lens = [], []
    for row in rows:
        messages = build_messages(row, schema_ddl_)[:-1]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_texts.append(text)
        real_prompt_lens.append(len(tokenizer(text, add_special_tokens=False)['input_ids']))

    # Left-padding is required for batched generation (every row must start generating at the
    # same index) -- handled entirely within this call and stripped back out immediately after,
    # so nothing downstream ever has to reason about left-padded position ids.
    prev_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    enc = tokenizer(prompt_texts, return_tensors='pt', padding=True, add_special_tokens=False).to(DEVICE)
    tokenizer.padding_side = prev_padding_side
    padded_prompt_len = enc['input_ids'].shape[1]

    rep_input_ids = enc['input_ids'].repeat_interleave(group_size, dim=0)
    rep_attn = enc['attention_mask'].repeat_interleave(group_size, dim=0)

    with amp_ctx():
        gen = model.generate(
            input_ids=rep_input_ids, attention_mask=rep_attn,
            max_new_tokens=cfg['max_new_tokens_rollout'],
            do_sample=True, temperature=cfg['rollout_temperature'], top_p=cfg['rollout_top_p'],
            pad_token_id=tokenizer.eos_token_id,
        )   # [n_prompts * group_size, padded_prompt_len + completion_len]

    trimmed, pred_texts = [], []
    for i in range(gen.shape[0]):
        p = i // group_size
        left_pad = padded_prompt_len - real_prompt_lens[p]
        completion_ids_full = gen[i][padded_prompt_len:]
        eos_pos = (completion_ids_full == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
        end = (eos_pos[0].item() + 1) if len(eos_pos) > 0 else len(completion_ids_full)
        # Strip this row's own left-padding, keeping [real prompt tokens + trimmed completion] --
        # the same padding-free per-sequence shape the original unbatched code produced, so
        # everything downstream (old-logprob batching, response_mask, training) is unchanged.
        full_ids = gen[i][left_pad: padded_prompt_len + end]
        pred_text = tokenizer.decode(gen[i][padded_prompt_len: padded_prompt_len + end], skip_special_tokens=True).strip()
        trimmed.append(full_ids)
        pred_texts.append(pred_text)

    rewards_flat = [compute_reward(pred_texts[i], rows[i // group_size]['gold_sql'], duckdb_con_, cfg)
                     for i in range(gen.shape[0])]

    # Batched old-logprob forward pass across all n_prompts * group_size completions -- same
    # right-pad + real attention_mask pattern as the earlier per-group fix, just over a bigger
    # batch now. No left-padding here (already stripped above), so real tokens occupy [0, L) with
    # natural position ids -- no special position_id handling needed.
    max_len = max(t.shape[0] for t in trimmed)
    pad_id = tokenizer.pad_token_id
    batch_ids = torch.full((len(trimmed), max_len), pad_id, dtype=torch.long, device=DEVICE)
    attn2 = torch.zeros((len(trimmed), max_len), dtype=torch.long, device=DEVICE)
    response_masks = torch.zeros((len(trimmed), max_len), dtype=torch.long)
    for i, full_ids in enumerate(trimmed):
        p = i // group_size
        L = full_ids.shape[0]
        batch_ids[i, :L] = full_ids
        attn2[i, :L] = 1
        response_masks[i, real_prompt_lens[p]:L] = 1

    with amp_ctx():
        out = model(input_ids=batch_ids, attention_mask=attn2)
    old_logprobs_batch = gather_token_logprobs(out.logits, batch_ids).float().cpu()

    groups = []
    for p in range(n_prompts):
        samples = []
        for g in range(group_size):
            i = p * group_size + g
            L = trimmed[i].shape[0]
            samples.append({'input_ids': trimmed[i].cpu(), 'response_mask': response_masks[i, :L],
                             'old_logprobs': old_logprobs_batch[i, :L - 1], 'pred_text': pred_texts[i]})
        rewards = torch.tensor(rewards_flat[p * group_size:(p + 1) * group_size], dtype=torch.float32)
        groups.append({'samples': samples, 'rewards': rewards, 'row': rows[p]})
    return groups


def collect_rollout_batch(model, tokenizer, sampler, schema_ddl_, cfg, duckdb_con_):
    target = cfg['rollout_groups_per_batch']
    kept = []
    stats = {'groups_sampled': 0, 'groups_kept': 0, 'rewards_seen': []}
    attempts = 0
    batch_n = cfg['rollout_prompts_per_batch']
    while len(kept) < target and attempts < cfg['max_rollout_attempts']:
        n = min(batch_n, cfg['max_rollout_attempts'] - attempts)
        rows = sampler.sample(n)
        groups = sample_group_batch(model, tokenizer, rows, schema_ddl_, cfg, duckdb_con_)
        for grp in groups:
            stats['groups_sampled'] += 1
            stats['rewards_seen'].extend(grp['rewards'].tolist())
            if bool(keep_groups(grp['rewards'], cfg['group_size']).item()):
                advs = group_advantages(grp['rewards'], cfg['group_size'])
                kept.append({'samples': grp['samples'], 'rewards': grp['rewards'], 'advantages': advs, 'row': grp['row']})
                stats['groups_kept'] += 1
        attempts += n
    return kept, stats

## Trainer

One `RLTrainer` per seed, starting from that seed's SFT `best/` checkpoint (`PeftModel.from_pretrained`
on the same already-loaded 4-bit base). Resumable the same way as SFT: `last/` + `training_state.pt`
checked on init, `best/` updated only on evaluation steps.

**Global token-level normalization under per-sample accumulation:** `compute_dapo_loss` normalizes
by the token count of whatever batch it's given. Rather than pad every sample in a rollout batch
into one tensor (expensive -- rollout batches mix very different lengths), each sample is run
through its own forward pass and the returned per-sample loss is rescaled by
`(that sample's token count) / (total token count across the whole kept batch)` before
`.backward()`. Since `.backward()` calls accumulate gradients additively, the sum of these
rescaled per-sample backward passes reproduces the gradient of the single globally-normalized
loss DAPO specifies -- without ever materializing a padded batch tensor.

In [ ]:
def _pad_chunk(chunk, pad_token_id):
    # chunk: list of (sample_dict, advantage) -- sample_dict has input_ids/response_mask/old_logprobs
    max_len = max(s['input_ids'].shape[0] for s, _ in chunk)
    bs = len(chunk)
    ids = torch.full((bs, max_len), pad_token_id, dtype=torch.long, device=DEVICE)
    attn = torch.zeros((bs, max_len), dtype=torch.long, device=DEVICE)
    mask = torch.zeros((bs, max_len - 1), dtype=torch.float32, device=DEVICE)
    old_lp = torch.zeros((bs, max_len - 1), dtype=torch.float32, device=DEVICE)
    for i, (s, _) in enumerate(chunk):
        L = s['input_ids'].shape[0]
        ids[i, :L] = s['input_ids'].to(DEVICE)
        attn[i, :L] = 1
        mask[i, :L - 1] = s['response_mask'][1:].float().to(DEVICE)
        old_lp[i, :L - 1] = s['old_logprobs'].to(DEVICE)
    adv_t = torch.tensor([a for _, a in chunk], device=DEVICE, dtype=torch.float32)
    return ids, attn, mask, old_lp, adv_t


def _dapo_backward_chunk(model, tokenizer, chunk, cfg, global_token_count):
    '''Pads one micro-batch of (sample, advantage) pairs, runs the DAPO forward+loss, and calls
    .backward() rescaled by this chunk's share of global_token_count -- summing rescaled backward
    passes across several chunks reproduces the gradient of DAPO's single globally-normalized loss
    (same trick as the original per-sample loop, just applied per-chunk instead of per-sample; see
    METHODOLOGY_LOG.md, "Global token-level normalization under per-sample gradient accumulation").
    Shared by RLTrainer._train_step and probe_train_micro_batch_size so both use one code path.'''
    ids, attn, mask, old_lp, adv_t = _pad_chunk(chunk, tokenizer.pad_token_id)
    with amp_ctx():
        out = model(input_ids=ids, attention_mask=attn)
        new_lp = gather_token_logprobs(out.logits, ids)
    chunk_loss = compute_dapo_loss(new_lp, old_lp, adv_t, mask,
                                    eps_low=cfg['eps_low'], eps_high=cfg['eps_high'])
    n_chunk = int(mask.sum().item())
    (chunk_loss * (n_chunk / max(1, global_token_count))).backward()
    return n_chunk


class RLTrainer:
    def __init__(self, cfg, seed, sft_best_dir, prompt_pool, dev_rows_, schema_ddl_, out_dir, duckdb_con_, eval_sample):
        set_seed(seed)
        self.cfg = cfg; self.seed = seed; self.out_dir = out_dir
        self.schema_ddl = schema_ddl_; self.duckdb_con = duckdb_con_
        self.prompt_pool = prompt_pool; self.eval_sample = eval_sample
        os.makedirs(out_dir, exist_ok=True)
        self._tee = TeeFile(sys.stdout, os.path.join(out_dir, 'train.log'))

        self.sampler = CyclicPromptSampler(prompt_pool, seed)
        self.sampler.on_reshuffle = lambda n: self._log(
            f'[sampler] cycle {n} starting -- full {len(prompt_pool)}-row pool reshuffled, '
            f'every instance sampled at least once so far')

        self.model, self.tokenizer = self._build_model_and_tokenizer(sft_best_dir)
        self.model.to(DEVICE)
        self.model = _maybe_compile(self.model, cfg)
        self.trainable_params = [p for p in self.model.parameters() if p.requires_grad]
        self.opt = torch.optim.AdamW(self.trainable_params, lr=cfg['lr'], weight_decay=cfg['weight_decay'])

        self.history = {'mean_reward': [], 'groups_kept_frac': [],
                         'gen_exact_match': [], 'gen_exec_match': [],
                         'gen_abstention_precision': [], 'gen_abstention_recall': []}
        self.best_score = -float('inf'); self.best_step = 0; self.start_step = 0
        self.checks_since_improvement = 0
        self._maybe_resume_training_state()

    def _build_model_and_tokenizer(self, sft_best_dir):
        wrapper = FHIRSQLLLM(self.cfg)
        tokenizer = wrapper._load_tokenizer()
        base_model = wrapper._load_base_model()
        last_dir = os.path.join(self.out_dir, 'last')
        last_prev_dir = last_dir + '.prev'
        if os.path.isdir(last_dir):
            self._log(f'[resume] loading RL adapter weights from {last_dir}')
            model = PeftModel.from_pretrained(base_model, last_dir, is_trainable=True)
        elif os.path.isdir(last_prev_dir):
            self._log(f'[resume] {last_dir} missing (crash mid-rotation?) -- '
                      f'falling back to {last_prev_dir}')
            model = PeftModel.from_pretrained(base_model, last_prev_dir, is_trainable=True)
        else:
            self._log(f'[start] loading SFT best checkpoint as RL starting point: {sft_best_dir}')
            model = PeftModel.from_pretrained(base_model, sft_best_dir, is_trainable=True)
        wrapper.inspect_trainable_parameters(model)
        return model, tokenizer

    def _log(self, msg):
        print(msg, file=self._tee)

    def _state_path(self):
        return os.path.join(self.out_dir, 'training_state.pt')

    def _maybe_resume_training_state(self):
        sp = self._state_path()
        if not os.path.exists(sp):
            return
        state = torch.load(sp, map_location=DEVICE)
        self.opt.load_state_dict(state['optimizer'])
        self.history = state['history']
        self.best_score = state['best_score']; self.best_step = state['best_step']
        self.start_step = state['step']
        if 'sampler' in state:
            self.sampler.load_state_dict(state['sampler'])
            self._log(f'[resume] sampler state restored (cycle {self.sampler.cycle_count}, '
                      f'position {self.sampler._pos}/{len(self.sampler.prompt_pool)})')
        self.checks_since_improvement = state.get('checks_since_improvement', 0)
        self._log(f'[resume] training state loaded, resuming from step {self.start_step + 1}')

    def _save(self, step, is_best):
        _atomic_save_adapter(self.model, os.path.join(self.out_dir, 'last'))
        if is_best:
            _atomic_save_adapter(self.model, os.path.join(self.out_dir, 'best'))
        _atomic_torch_save({
            'optimizer': self.opt.state_dict(), 'step': step, 'history': self.history,
            'best_score': self.best_score, 'best_step': self.best_step,
            'sampler': self.sampler.state_dict(),
            'checks_since_improvement': self.checks_since_improvement,
        }, self._state_path())
        json.dump(self.history, open(os.path.join(self.out_dir, 'history.json'), 'w'), indent=2)

    def _train_step(self):
        kept, stats = collect_rollout_batch(self.model, self.tokenizer, self.sampler,
                                             self.schema_ddl, self.cfg, self.duckdb_con)
        if not kept:
            return stats, None, None

        g0 = kept[0]
        peek = {'question': g0['row']['question'], 'gold': g0['row']['gold_sql'],
                'pred': g0['samples'][0]['pred_text'], 'reward': g0['rewards'][0].item()}

        items = []
        for group in kept:
            for s, r, a in zip(group['samples'], group['rewards'].tolist(), group['advantages'].tolist()):
                items.append((s, r, a))
        global_token_count = sum(int(s['response_mask'].sum().item()) for s, _, _ in items)
        mb_size = self.cfg['train_micro_batch_size']

        for _ in range(self.cfg['ppo_epochs_per_rollout']):
            self.model.train()
            self.opt.zero_grad(set_to_none=True)
            for start in range(0, len(items), mb_size):
                chunk = [(s, a) for s, r, a in items[start:start + mb_size]]
                _dapo_backward_chunk(self.model, self.tokenizer, chunk, self.cfg, global_token_count)
            torch.nn.utils.clip_grad_norm_(self.trainable_params, 1.0)
            self.opt.step()

        mean_reward = sum(r for _, r, _ in items) / len(items)
        return stats, mean_reward, peek

    def train(self):
        for step in range(self.start_step, self.cfg['num_rollout_steps']):
            t0 = time.time()
            stats, mean_reward, peek = self._train_step()
            kept_frac = stats['groups_kept'] / max(1, stats['groups_sampled'])
            self.history['mean_reward'].append(mean_reward if mean_reward is not None else float('nan'))
            self.history['groups_kept_frac'].append(kept_frac)

            gm = {'gen_exact_match': float('nan'), 'gen_exec_match': float('nan'),
                  'gen_abstention_precision': float('nan'), 'gen_abstention_recall': float('nan')}
            if (step + 1) % self.cfg['eval_every_n_steps'] == 0:
                gm = evaluate_generation(self.model, self.tokenizer, self.eval_sample,
                                          self.schema_ddl, self.cfg, self.duckdb_con)
            for k in ('gen_exact_match', 'gen_exec_match', 'gen_abstention_precision', 'gen_abstention_recall'):
                self.history[k].append(gm[k])

            score = gm['gen_exec_match']
            was_eval_step = (step + 1) % self.cfg['eval_every_n_steps'] == 0
            is_best = (score == score) and score > self.best_score
            if is_best:
                self.best_score = score; self.best_step = step + 1
                self.checks_since_improvement = 0
            elif was_eval_step:
                self.checks_since_improvement += 1
            early_stop = self.checks_since_improvement >= self.cfg['early_stopping_patience']
            is_last_step = (step + 1) == self.cfg['num_rollout_steps'] or early_stop
            should_save = ((step + 1) % self.cfg['save_every_n_steps'] == 0) or is_best or is_last_step
            if should_save:
                self._save(step + 1, is_best)

            mr = mean_reward if mean_reward is not None else float('nan')
            self._log(f'[seed{self.seed}] step {step+1}/{self.cfg["num_rollout_steps"]} '
                      f'| reward {mr:.3f} | kept {stats["groups_kept"]}/{stats["groups_sampled"]} ({kept_frac:.0%}) '
                      f'| exec {gm["gen_exec_match"]:.3f} (best {self.best_score:.3f}@{self.best_step}) '
                      f'| exact {gm["gen_exact_match"]:.3f} | {time.time()-t0:.0f}s')
            if peek is not None:
                self._log(f'    peek | Q: {peek["question"][:100]!r}')
                self._log(f'    peek | gold: {peek["gold"][:120]!r}')
                self._log(f'    peek | pred: {peek["pred"][:120]!r}  (reward={peek["reward"]:.2f})')
            if early_stop:
                self._log(f'[seed{self.seed}] early stopping at step {step+1} -- no improvement in '
                          f'{self.checks_since_improvement} eval checks '
                          f'({self.checks_since_improvement * self.cfg["eval_every_n_steps"]} steps), '
                          f'best exec-match {self.best_score:.3f} @ step {self.best_step}')
                break
        self._tee.close()
        return self.best_score, self.best_step

## Multi-seed driver

Runs one RL pass per SFT seed, each starting from `sft_outputs/seed_<seed>/best/`. Fails loudly
up front if any seed's SFT checkpoint is missing, rather than silently skipping a seed.

In [ ]:
def seed_out_dir(seed):
    return os.path.join(OUT_BASE, f'seed_{seed}')


def seed_is_done(seed):
    hp = os.path.join(seed_out_dir(seed), 'history.json')
    if not os.path.exists(hp):
        return False
    h = json.load(open(hp))
    if len(h['mean_reward']) >= CFG['num_rollout_steps']:
        return True
    # Early stopping means a seed can finish without ever reaching num_rollout_steps --
    # without this check, seed_is_done would wrongly say False, and the driver would reload the
    # model just to immediately re-trigger early stopping on the next eval check.
    sp = os.path.join(seed_out_dir(seed), 'training_state.pt')
    if os.path.exists(sp):
        state = torch.load(sp, map_location='cpu')
        if state.get('checks_since_improvement', 0) >= CFG['early_stopping_patience']:
            return True
    return False


sft_best_dirs = {seed: os.path.join(SFT_OUT_BASE, f'seed_{seed}', 'best') for seed in CFG['seeds']}
for seed, d in sft_best_dirs.items():
    assert os.path.isdir(d), f'no SFT best checkpoint for seed {seed} at {d} -- run sft_train.ipynb first'

GRID_T0 = time.time()
for seed in CFG['seeds']:
    if seed_is_done(seed):
        print(f'[skip] seed {seed} already complete'); continue
    print(f'\n================  seed {seed}  ================')
    t0 = time.time()
    trainer = RLTrainer(CFG, seed, sft_best_dirs[seed], train_rows, dev_rows, schema_ddl,
                         seed_out_dir(seed), duckdb_con, GEN_EVAL_SAMPLE)
    best_score, best_step = trainer.train()
    print(f'---- seed {seed} done in {(time.time()-t0)/60:.1f} min | best exec-match {best_score:.3f} @ step {best_step} ----')
    del trainer
    gc.collect()   # del alone doesn't guarantee immediate release of CUDA-referenced
                    # tensors (optimizer state, etc.) -- without this, fragmentation from
                    # one seed's run can carry into the next seed's OOM budget even though
                    # a probe measured a safe micro_batch_size on a clean memory state
                    # (observed: OOM on one seed at the probe's 2nd-best size).
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
print(f'\n=====  all seeds complete in {(time.time()-GRID_T0)/3600:.2f} h  =====')

[skip] seed 42 already complete
[skip] seed 43 already complete
[skip] seed 44 already complete

=====  all seeds complete in 0.01 h  =====


## Progress (safe to run anytime, even mid-training)

In [ ]:
print(f"{'seed':<6} {'steps_done':<12} {'best_exec_match':<16} {'best_step':<10}")
for seed in CFG['seeds']:
    hp = os.path.join(seed_out_dir(seed), 'history.json')
    if not os.path.exists(hp):
        print(f'{seed:<6} not started'); continue
    h = json.load(open(hp))
    n = len(h['mean_reward'])
    steps_str = f"{n}/{CFG['num_rollout_steps']}"
    exec_vals = [v for v in h['gen_exec_match'] if v == v]
    best = max(exec_vals) if exec_vals else float('nan')
    best_step = (h['gen_exec_match'].index(best) + 1) if exec_vals else 0
    print(f'{seed:<6} {steps_str:<12} {best:<16.3f} {best_step:<10}')

seed   steps_done   best_exec_match  best_step 
42     200/200      0.964            20        
43     160/200      0.964            60        
44     120/200      0.964            20        


## Final comparison: RL vs SFT, on the held-out test split

Loads each seed's already-computed `sft_outputs/seed_<seed>/test_eval.json` (no need to
recompute -- SFT already produced it) as the baseline, and evaluates each seed's RL `best/`
checkpoint on the identical `TEST_GEN_SAMPLE`/`test_rows` for a same-rows comparison, exactly
mirroring how SFT compared itself against the frozen base model.

In [ ]:
def evaluate_checkpoint_on_test(model, adapter_dir, adapter_name, tokenizer, test_loader_, test_gen_sample_, schema_ddl_, cfg, duckdb_con_):
    model.load_adapter(adapter_dir, adapter_name=adapter_name)
    prev_adapter = model.active_adapter
    model.set_adapter(adapter_name)
    model.eval()
    fast = evaluate_loss_and_token_acc(model, test_loader_, amp_ctx)
    gen = evaluate_generation(model, tokenizer, test_gen_sample_, schema_ddl_, cfg, duckdb_con_)
    model.set_adapter(prev_adapter)
    model.delete_adapter(adapter_name)
    return {'val_loss': fast['loss'], 'val_perplexity': fast['perplexity'], 'val_token_acc': fast['token_acc'], **gen}


rl_test_results = {}
for seed in CFG['seeds']:
    sft_test_path = os.path.join(SFT_OUT_BASE, f'seed_{seed}', 'test_eval.json')
    rl_test_path = os.path.join(seed_out_dir(seed), 'test_eval.json')
    rl_best_dir = os.path.join(seed_out_dir(seed), 'best')
    if not (os.path.exists(sft_test_path) and os.path.isdir(rl_best_dir)):
        print(f'[skip] seed {seed}: missing SFT test_eval.json or RL best/ checkpoint'); continue

    if not os.path.exists(rl_test_path):
        wrapper = FHIRSQLLLM(CFG)
        tokenizer = wrapper._load_tokenizer()
        base_model = wrapper._load_base_model()
        model = PeftModel.from_pretrained(base_model, rl_best_dir, is_trainable=False)
        model.to(DEVICE)
        test_loader = make_loader(test_rows, tokenizer, schema_ddl, CFG, shuffle=False)
        result = evaluate_checkpoint_on_test(model, rl_best_dir, 'rl_test_eval_ckpt', tokenizer,
                                              test_loader, TEST_GEN_SAMPLE, schema_ddl, CFG, duckdb_con)
        json.dump(result, open(rl_test_path, 'w'), indent=2)
        del model, base_model
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    rl_test_results[seed] = json.load(open(rl_test_path))
    print(f'seed {seed} RL test metrics:', json.dumps(rl_test_results[seed], indent=2))

sft_test_results = {}
for seed in CFG['seeds']:
    p = os.path.join(SFT_OUT_BASE, f'seed_{seed}', 'test_eval.json')
    if os.path.exists(p):
        sft_test_results[seed] = json.load(open(p))

print('\nseed   metric          SFT       RL        delta (positive = RL better)')
for seed in CFG['seeds']:
    if seed not in rl_test_results or seed not in sft_test_results:
        continue
    sft_r, rl_r = sft_test_results[seed], rl_test_results[seed]
    for metric in ('gen_exec_match', 'gen_exact_match', 'gen_abstention_precision', 'gen_abstention_recall'):
        print(f'{seed:<6} {metric:<15} {sft_r[metric]:<9.3f} {rl_r[metric]:<9.3f} {rl_r[metric]-sft_r[metric]:+.3f}')

seed 42 RL test metrics: {
  "val_loss": 0.011831894615279543,
  "val_perplexity": 1.0119021683640799,
  "val_token_acc": 0.9985092340647252,
  "gen_exact_match": 0.9821428571428571,
  "gen_exec_match": 0.9107142857142857,
  "gen_abstention_precision": 1.0,
  "gen_abstention_recall": 1.0,
  "gen_efficiency_speedup": 1.0036570285074025
}
seed 43 RL test metrics: {
  "val_loss": 0.010960413495224556,
  "val_perplexity": 1.011020698876804,
  "val_token_acc": 0.9985816013431367,
  "gen_exact_match": 0.9821428571428571,
  "gen_exec_match": 0.9107142857142857,
  "gen_abstention_precision": 1.0,
  "gen_abstention_recall": 1.0,
  "gen_efficiency_speedup": 1.014947829195649
}
seed 44 RL test metrics: {
  "val_loss": 0.011297651318851413,
  "val_perplexity": 1.0113617107947614,
  "val_token_acc": 0.9985526544317721,
  "gen_exact_match": 0.9821428571428571,
  "gen_exec_match": 0.8928571428571429,
  "gen_abstention_precision": 1.0,
  "gen_abstention_recall": 1.0,
  "gen_efficiency_speedup": 0.9475

## Three-way ablation: frozen base vs SFT-only vs SFT+RL

Joins `sft_outputs/test_baseline.json` (frozen base model, computed once -- identical for every
seed by construction, since it's deterministic under greedy decoding) with each seed's
`sft_outputs/seed_<seed>/test_eval.json` (SFT-only) and `rl_test_results` above (SFT+RL), on the
same held-out test split and fixed `TEST_GEN_SAMPLE`. This is the DoRA-only-vs-+DAPO ablation.

`gen_efficiency_speedup` will show as `nan` for the `frozen` and `SFT-only` rows -- that metric was
added to `evaluate_generation` after `sft_train.ipynb`'s `test_baseline.json`/`test_eval.json`
files were already computed, and `sft_train.ipynb` is intentionally not being touched to backfill
it while SFT training is actively running. Only `SFT+RL` carries a real measured value for it.

To run the correctness-only vs. correctness+efficiency ablation this notebook's `CFG['run_name']`
is set up for: rerun the driver with `CFG['efficiency_bonus_max'] = 0` and a different
`CFG['run_name']` (e.g. `'correctness_only'`), then compare that run's `rl_test_results` against
this one's -- both will have real `gen_efficiency_speedup` numbers, computed identically, so the
comparison is apples-to-apples.

In [ ]:
frozen_baseline_path = os.path.join(SFT_OUT_BASE, 'test_baseline.json')
frozen_baseline = json.load(open(frozen_baseline_path)) if os.path.exists(frozen_baseline_path) else None

ABLATION_METRICS = ['gen_exec_match', 'gen_exact_match', 'gen_abstention_precision',
                     'gen_abstention_recall', 'gen_efficiency_speedup']


def fmt_row(label, r):
    if r is None:
        return f'{label:<10} (missing)'
    return f'{label:<10} ' + ' '.join(f'{r.get(m, float("nan")):<10.3f}' for m in ABLATION_METRICS)


print(' ' * 11 + ' '.join(f'{m:<10}' for m in ABLATION_METRICS))
print(fmt_row('frozen', frozen_baseline), ' (same for every seed -- computed once)')
print()
for seed in CFG['seeds']:
    print(f'seed {seed}:')
    print(' ', fmt_row('SFT-only', sft_test_results.get(seed)))
    print(' ', fmt_row('SFT+RL', rl_test_results.get(seed)))
print()
print("gen_efficiency_speedup is nan for 'frozen'/'SFT-only' -- see markdown above for why. "
      "Only 'SFT+RL' has a real measured value.")

           gen_exec_match gen_exact_match gen_abstention_precision gen_abstention_recall gen_efficiency_speedup
frozen     0.000      0.000      0.222      1.000      nan         (same for every seed -- computed once)

seed 42:
  SFT-only   0.929      0.982      1.000      1.000      nan       
  SFT+RL     0.911      0.982      1.000      1.000      1.004     
seed 43:
  SFT-only   0.911      0.982      1.000      1.000      nan       
  SFT+RL     0.911      0.982      1.000      1.000      1.015     
seed 44:
  SFT-only   0.893      0.982      1.000      1.000      nan       
  SFT+RL     0.893      0.982      1.000      1.000      0.948     

gen_efficiency_speedup is nan for 'frozen'/'SFT-only' -- see markdown above for why. Only 'SFT+RL' has a real measured value.


## Heldout benchmark: 3-way x 2-arm comparison (the central research objective)

This is the comparison everything else in this project has been building toward -- see
METHODOLOGY_LOG.md, "Evaluation objective."
Unlike the `test` split above (in-corpus, same patient population every training row came from),
`heldout.duckdb` is a disjoint 6,383-patient population no training data was ever built from.

Two concept arms, evaluated separately, so a generalization *gap* (not just a single number) comes out:
- **familiar**: archetypes built from the 382 concepts used in all training data --
  isolates *population* generalization (same concepts, unfamiliar patients).
- **unseen**: archetypes built from the 87 concepts that exist in heldout's data but never appear
  anywhere in train's -- isolates *concept* generalization (genuinely novel codes). Necessarily
  narrower in scope than `familiar` (tier-1/2 patient-level archetypes on condition/procedure/
  medication_request only) since all 87 concepts have just 1-3 patients each -- see
  `scripts/generate_heldout_gold.py`'s docstring for why population-aggregate archetypes aren't
  possible here. This is a real property of the data, not a shortcut taken in building the benchmark.

For each arm: frozen base (no adapter, computed once -- identical across seeds under greedy
decoding) vs. each seed's SFT-only checkpoint vs. each seed's SFT+RL checkpoint, on the same
accuracy/perplexity/efficiency metrics used above. Results are cached to disk (frozen once,
SFT-only per seed, SFT+RL per seed x run_name) so re-running this notebook doesn't repeat
expensive generation. Safe to run anytime -- skips whatever checkpoints don't exist yet.

In [ ]:
HELDOUT_EVAL_DIR = os.path.join(RL_OUTPUTS_ROOT, 'heldout_eval')   # shared across ablation runs -- frozen/SFT-only don't depend on run_name
os.makedirs(HELDOUT_EVAL_DIR, exist_ok=True)

heldout_con = duckdb.connect(LOCAL_HELDOUT_DUCKDB, read_only=True)

HELDOUT_ROWS = {}
HELDOUT_GEN_SAMPLE = {}
HELDOUT_LOADERS_NEED_TOKENIZER = {}   # built lazily below once a tokenizer is loaded, per arm
for arm, path in LOCAL_HELDOUT_JSONL.items():
    rows_ = [json.loads(l) for l in open(path, encoding='utf-8')]
    HELDOUT_ROWS[arm] = rows_
    HELDOUT_GEN_SAMPLE[arm] = random.Random(CFG['split_seed']).sample(
        rows_, min(CFG['heldout_eval_gen_sample_size'], len(rows_))
    )
    print(f"heldout[{arm}]: {len(rows_)} rows, gen-eval sample = {len(HELDOUT_GEN_SAMPLE[arm])}")


def evaluate_state_on_heldout(model, tokenizer, cfg, label=None):
    '''Runs both arms' teacher-forced (loss/perplexity/token-acc) + generation (exact/exec-match,
    abstention precision/recall, efficiency speedup) eval for whatever adapter state `model` is
    currently in (frozen base, or a loaded/activated PEFT adapter) -- caller is responsible for
    adapter state, this function only evaluates.

    label: if given, failing rows for each arm are logged to
    HELDOUT_EVAL_DIR/failures/{label}_{arm}.jsonl as they're found -- None (default) skips
    logging entirely. Cached (already-computed) legs never re-run, so this only actually
    collects logs for whichever legs are computed fresh.'''
    out = {}
    for arm in ('familiar', 'unseen'):
        loader = make_loader(HELDOUT_ROWS[arm], tokenizer, schema_ddl, cfg, shuffle=False)
        fast = evaluate_loss_and_token_acc(model, loader, amp_ctx)
        failure_log_path = None
        if label is not None:
            failure_dir = os.path.join(HELDOUT_EVAL_DIR, 'failures')
            os.makedirs(failure_dir, exist_ok=True)
            failure_log_path = os.path.join(failure_dir, f'{label}_{arm}.jsonl')
        gen = evaluate_generation(model, tokenizer, HELDOUT_GEN_SAMPLE[arm], schema_ddl, cfg, heldout_con,
                                   failure_log_path=failure_log_path)
        out[arm] = {'val_loss': fast['loss'], 'val_perplexity': fast['perplexity'], 'val_token_acc': fast['token_acc'], **gen}
    return out


if CFG['run_heldout_eval']:
    # ---- Frozen baseline on heldout (computed once -- deterministic under greedy decoding, same for every seed) ----
    frozen_heldout_paths = {arm: os.path.join(HELDOUT_EVAL_DIR, f'frozen_{arm}.json') for arm in ('familiar', 'unseen')}
    if not all(os.path.exists(p) for p in frozen_heldout_paths.values()):
        wrapper = FHIRSQLLLM(CFG)
        tokenizer = wrapper._load_tokenizer()
        base_model = wrapper._load_base_model()
        base_model.to(DEVICE)
        frozen_heldout = evaluate_state_on_heldout(base_model, tokenizer, CFG, label='frozen')
        for arm, result in frozen_heldout.items():
            json.dump(result, open(frozen_heldout_paths[arm], 'w'), indent=2)
        del base_model
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    frozen_heldout = {arm: json.load(open(p)) for arm, p in frozen_heldout_paths.items()}
    print('frozen (heldout):', json.dumps(frozen_heldout, indent=2))


    # ---- SFT-only and SFT+RL on heldout, per seed ----
    sft_heldout_results, rl_heldout_results = {}, {}
    for seed in CFG['seeds']:
        sft_best_dir = os.path.join(SFT_OUT_BASE, f'seed_{seed}', 'best')
        rl_best_dir = os.path.join(seed_out_dir(seed), 'best')

        sft_paths = {arm: os.path.join(HELDOUT_EVAL_DIR, f'sft_seed{seed}_{arm}.json') for arm in ('familiar', 'unseen')}
        if os.path.isdir(sft_best_dir) and not all(os.path.exists(p) for p in sft_paths.values()):
            print(f'[seed {seed}] SFT-only heldout eval starting...')
            _leg_t0 = time.time()
            wrapper = FHIRSQLLLM(CFG)
            tokenizer = wrapper._load_tokenizer()
            base_model = wrapper._load_base_model()
            model = PeftModel.from_pretrained(base_model, sft_best_dir, is_trainable=False)
            model.to(DEVICE)
            result = evaluate_state_on_heldout(model, tokenizer, CFG, label=f'sft_seed{seed}')
            for arm, r in result.items():
                json.dump(r, open(sft_paths[arm], 'w'), indent=2)
            del model, base_model
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
            print(f'[seed {seed}] SFT-only heldout eval done in {(time.time()-_leg_t0)/60:.1f} min')
        if all(os.path.exists(p) for p in sft_paths.values()):
            sft_heldout_results[seed] = {arm: json.load(open(p)) for arm, p in sft_paths.items()}
            print(f'[seed {seed}] SFT-only (cached or just-computed):', json.dumps(sft_heldout_results[seed], indent=2))

        rl_paths = {arm: os.path.join(OUT_BASE, 'heldout_eval', f'rl_seed{seed}_{arm}.json') for arm in ('familiar', 'unseen')}
        os.makedirs(os.path.join(OUT_BASE, 'heldout_eval'), exist_ok=True)
        if os.path.isdir(rl_best_dir) and not all(os.path.exists(p) for p in rl_paths.values()):
            print(f'[seed {seed}] SFT+RL heldout eval starting...')
            _leg_t0 = time.time()
            wrapper = FHIRSQLLLM(CFG)
            tokenizer = wrapper._load_tokenizer()
            base_model = wrapper._load_base_model()
            model = PeftModel.from_pretrained(base_model, rl_best_dir, is_trainable=False)
            model.to(DEVICE)
            result = evaluate_state_on_heldout(model, tokenizer, CFG, label=f'rl_seed{seed}')
            for arm, r in result.items():
                json.dump(r, open(rl_paths[arm], 'w'), indent=2)
            del model, base_model
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
            print(f'[seed {seed}] SFT+RL heldout eval done in {(time.time()-_leg_t0)/60:.1f} min')
        if all(os.path.exists(p) for p in rl_paths.values()):
            rl_heldout_results[seed] = {arm: json.load(open(p)) for arm, p in rl_paths.items()}
            print(f'[seed {seed}] SFT+RL (cached or just-computed):', json.dumps(rl_heldout_results[seed], indent=2))

        if seed not in sft_heldout_results and seed not in rl_heldout_results:
            print(f'[skip] seed {seed}: no SFT best/ or RL best/ checkpoint found yet')
else:
    frozen_heldout, sft_heldout_results, rl_heldout_results = {}, {}, {}
    print("[skip] run_heldout_eval=False -- not computing/loading heldout results")

heldout[familiar]: 10088 rows, gen-eval sample = 10088
heldout[unseen]: 2196 rows, gen-eval sample = 2196
frozen (heldout): {
  "familiar": {
    "val_loss": 0.9549367765457278,
    "val_perplexity": 2.598506291182354,
    "val_token_acc": 0.7815199007779907,
    "gen_exact_match": 0.0,
    "gen_exec_match": 0.0,
    "gen_abstention_precision": 0.42105263157894735,
    "gen_abstention_recall": 1.0,
    "gen_efficiency_speedup": NaN
  },
  "unseen": {
    "val_loss": 0.6611321701042047,
    "val_perplexity": 1.9369840889262424,
    "val_token_acc": 0.7686932405242264,
    "gen_exact_match": 0.0,
    "gen_exec_match": 0.0,
    "gen_abstention_precision": 0.8571428571428571,
    "gen_abstention_recall": 0.8571428571428571,
    "gen_efficiency_speedup": NaN
  }
}


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

In [ ]:
if CFG['run_heldout_eval']:
    HELDOUT_METRICS = ['val_perplexity', 'gen_exec_match', 'gen_exact_match', 'gen_structure_match',
                        'gen_abstention_precision', 'gen_abstention_recall', 'gen_efficiency_speedup']


    def fmt_heldout_row(label, r):
        if r is None:
            return f'{label:<10} (missing)'
        return f'{label:<10} ' + ' '.join(f'{r.get(m, float("nan")):<10.3f}' for m in HELDOUT_METRICS)


    for arm in ('familiar', 'unseen'):
        print(f'\n=== heldout[{arm}] ===')
        print(' ' * 11 + ' '.join(f'{m:<10}' for m in HELDOUT_METRICS))
        print(fmt_heldout_row('frozen', frozen_heldout.get(arm)), ' (same for every seed)')
        for seed in CFG['seeds']:
            print(f'seed {seed}:')
            print(' ', fmt_heldout_row('SFT-only', sft_heldout_results.get(seed, {}).get(arm)))
            print(' ', fmt_heldout_row('SFT+RL', rl_heldout_results.get(seed, {}).get(arm)))

    print('\n=== generalization gap (familiar - unseen), positive = better on familiar concepts ===')
    print(f"{'state':<10} " + ' '.join(f'{m:<10}' for m in HELDOUT_METRICS))
    if frozen_heldout.get('familiar') and frozen_heldout.get('unseen'):
        gap = {m: frozen_heldout['familiar'].get(m, float('nan')) - frozen_heldout['unseen'].get(m, float('nan')) for m in HELDOUT_METRICS}
        print(fmt_heldout_row('frozen', gap))
    for seed in CFG['seeds']:
        for label, results in (('SFT-only', sft_heldout_results), ('SFT+RL', rl_heldout_results)):
            r = results.get(seed, {})
            if r.get('familiar') and r.get('unseen'):
                gap = {m: r['familiar'].get(m, float('nan')) - r['unseen'].get(m, float('nan')) for m in HELDOUT_METRICS}
                print(fmt_heldout_row(f'{label} s{seed}', gap))
    print("\nWhether SFT/RL narrows this gap vs. the frozen baseline's gap is the direct answer to "
          "'does training generalize to unseen concepts, or only get better at memorized ones.'")
    print("\nReading note: gen_exec_match is the primary correctness signal on BOTH arms -- gold "
          "SQL resolves terminology through the valuesets lookup CTE rather than embedding a "
          "literal code, so exec-match does not conflate SQL-writing skill with code recall. "
          "gen_structure_match/gen_exact_match are stricter text-similarity diagnostics; the "
          "former's code-literal masking is inert under this design. See PAPER.md Sec. 4.")
else:
    print('[skip] run_heldout_eval=False -- nothing to report')

## Diagnostic: table-selection + familiar-arm efficiency (frozen vs SFT vs RL, decoupled from code-recall)

The main heldout eval above already separates SQL-writing skill from code recall via
`gen_structure_match`. This supplementary diagnostic checks the same distinction a different way,
kept from an earlier iteration of this eval for cross-validation.

**Table-selection accuracy** checks the second question with the checkpoints already in hand, no
retraining: does predicted SQL reference the *same set of core tables* as gold, regardless of
whether the specific `code`/`system` filter value is right? Checked on **both arms separately**
(a combined sample would be ~82% familiar-weighted by population size and could silently hide a
real unseen-arm degradation). A model that learned real schema navigation should show a
table-match gap between frozen and SFT even where exact/exec-match is near zero (e.g. the unseen
arm) -- if SFT closes a meaningful chunk of that gap, that's evidence fine-tuning taught schema
reasoning, separable from the code-memorization confound.

**Familiar-arm SQL efficiency** (`gen_efficiency_speedup`, median gold-time/pred-time among rows
scored exec-correct -- same methodology used everywhere else this metric appears) is measured
frozen vs. SFT vs. SFT+RL in the same pass, answering "did RL make correct SQL faster" directly
on heldout data (distinct from the last cell's SFT-vs-RL-only comparison on the in-corpus test
split). The efficiency check is reported for the familiar arm; the unseen arm's speedup is also
computed and reported in the heldout eval above.

Both checks reuse `HELDOUT_GEN_SAMPLE` (750/arm, already built above regardless of
`CFG['run_heldout_eval']`) rather than a separate diagnostic-only sample. Runs on whatever
checkpoints currently exist (frozen always; SFT-only/SFT+RL per seed, skipped if that seed's
`best/` doesn't exist yet).

In [ ]:
_CORE_TABLES = {'patient', 'condition', 'observation', 'medication_request', 'encounter',
                'procedure', 'immunization', 'allergy', 'careplan', 'diagnostic_report', 'imaging_study'}


def _referenced_tables(sql):
    '''Which core tables does this SQL reference (FROM/JOIN), regardless of whether the filter
    values inside it are correct? UNANSWERABLE has no tables by construction, excluded by the
    caller rather than treated as a table-selection question.'''
    found = re.findall(r'\b(?:FROM|JOIN)\s+([a-zA-Z_][a-zA-Z0-9_]*)', sql, flags=re.IGNORECASE)
    return frozenset(t.lower() for t in found if t.lower() in _CORE_TABLES)


@torch.no_grad()
def evaluate_table_selection(model, tokenizer, sample_rows, schema_ddl_, cfg):
    '''table_match_rate: fraction of answerable rows where predicted SQL references the identical
    set of core tables as gold -- coarser than gen_exact_match/gen_exec_match, and specifically
    insensitive to whether the code/filter VALUE is right, only whether the model reasoned its way
    to the right TABLE(s). confusion: {(gold_tables, pred_tables): count} for inspecting which
    tables get confused for which. Batched the same way as evaluate_generation.'''
    model.eval()
    prev_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    batch_size = cfg.get('eval_gen_batch_size', cfg['rollout_prompts_per_batch'] * cfg['group_size'])

    n_table_match = n_scored = 0
    confusion = {}
    for start in range(0, len(sample_rows), batch_size):
        batch_rows = sample_rows[start:start + batch_size]
        prompt_texts = []
        for r in batch_rows:
            messages = build_messages(r, schema_ddl_)[:-1]
            prompt_texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        enc = tokenizer(prompt_texts, return_tensors='pt', padding=True, truncation=True,
                         max_length=cfg['max_seq_len'], add_special_tokens=False).to(DEVICE)
        padded_prompt_len = enc['input_ids'].shape[1]
        gen = model.generate(**enc, max_new_tokens=cfg.get('max_new_tokens_eval', 2048), do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
        for i, r in enumerate(batch_rows):
            completion = tokenizer.decode(gen[i][padded_prompt_len:], skip_special_tokens=True).strip()
            pred = extract_sql_from_completion(completion) or ''
            gold = r['gold_sql'].strip()
            if gold == ABSTENTION_TOKEN:
                continue   # no table to select for "this can't be answered"
            n_scored += 1
            gold_tables = _referenced_tables(gold)
            pred_tables = _referenced_tables(pred)
            if pred_tables == gold_tables:
                n_table_match += 1
            key = (tuple(sorted(gold_tables)), tuple(sorted(pred_tables)))
            confusion[key] = confusion.get(key, 0) + 1

    tokenizer.padding_side = prev_padding_side
    return {'table_match_rate': n_table_match / max(1, n_scored), 'n_scored': n_scored, 'confusion': confusion}


def print_top_confusions(result, top_n=10):
    wrong = {k: v for k, v in result['confusion'].items() if k[0] != k[1]}
    if not wrong:
        print('  (no table-selection errors in this sample)')
        return
    for (gold_t, pred_t), cnt in sorted(wrong.items(), key=lambda x: -x[1])[:top_n]:
        print(f'  gold={gold_t or "()"} -> pred={pred_t or "()"}: {cnt}x')


# Reuses HELDOUT_GEN_SAMPLE (750/arm, already built above regardless of run_heldout_eval) instead
# of a separate smaller sample -- one canonical eval sample for every diagnostic here, not a
# diluted one-off. Table-selection is checked on BOTH arms SEPARATELY, not combined into one
# number (a combined sample would be ~82% familiar-weighted by population size,
# which would silently hide whether table-selection also degrades on the unseen arm the way
# exact/exec-match does). Efficiency (gen_efficiency_speedup, median gold-time/pred-time among
# EXEC-CORRECT rows only) is checked on the familiar arm specifically, frozen vs. SFT vs. SFT+RL --
# the unseen arm's speedup is computed separately in the heldout eval above.
diag_results = {}


def _run_diagnostics(label, adapter_dir=None):
    wrapper = FHIRSQLLLM(CFG)
    tokenizer = wrapper._load_tokenizer()
    base_model = wrapper._load_base_model()
    model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=False) if adapter_dir else base_model
    model.to(DEVICE)

    table_fam = evaluate_table_selection(model, tokenizer, HELDOUT_GEN_SAMPLE['familiar'], schema_ddl, CFG)
    table_unseen = evaluate_table_selection(model, tokenizer, HELDOUT_GEN_SAMPLE['unseen'], schema_ddl, CFG)
    eff = evaluate_generation(model, tokenizer, HELDOUT_GEN_SAMPLE['familiar'], schema_ddl, CFG, heldout_con)

    diag_results[label] = {
        'table_match_familiar': table_fam['table_match_rate'],
        'table_match_unseen': table_unseen['table_match_rate'],
        'gen_exec_match_familiar': eff['gen_exec_match'],
        'gen_efficiency_speedup_familiar': eff['gen_efficiency_speedup'],
        '_table_confusion_familiar': table_fam['confusion'],
    }
    r = diag_results[label]
    print(f"{label:<14} table[fam]={r['table_match_familiar']:.3f} (n={table_fam['n_scored']})  "
          f"table[unseen]={r['table_match_unseen']:.3f} (n={table_unseen['n_scored']})  "
          f"exec[fam]={r['gen_exec_match_familiar']:.3f}  speedup[fam]={r['gen_efficiency_speedup_familiar']:.3f}")

    del model, base_model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


_run_diagnostics('frozen')

for seed in CFG['seeds']:
    sft_best_dir = os.path.join(SFT_OUT_BASE, f'seed_{seed}', 'best')
    if os.path.isdir(sft_best_dir):
        _run_diagnostics(f'sft_seed{seed}', sft_best_dir)
    else:
        print(f'[skip] sft_seed{seed}: no best/ checkpoint yet')

    rl_best_dir = os.path.join(seed_out_dir(seed), 'best')
    if os.path.isdir(rl_best_dir):
        _run_diagnostics(f'rl_seed{seed}', rl_best_dir)
    else:
        print(f'[skip] rl_seed{seed}: no best/ checkpoint yet')

print('\n=== diagnostic summary ===')
print(f"{'state':<14} {'table[fam]':<12} {'table[unseen]':<14} {'exec[fam]':<12} {'speedup[fam]':<12}")
for label, r in diag_results.items():
    print(f"{label:<14} {r['table_match_familiar']:<12.3f} {r['table_match_unseen']:<14.3f} "
          f"{r['gen_exec_match_familiar']:<12.3f} {r['gen_efficiency_speedup_familiar']:<12.3f}")

print("\nTop table-confusions for 'frozen', familiar arm (what it queries when it gets the table wrong):")
print_top_confusions({'confusion': diag_results['frozen']['_table_confusion_familiar']})

## Plot: reward and exec-match across RL steps, per seed

Saved to `rl_outputs/rl_progress.png`.

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = [('mean_reward', 'mean rollout reward'), ('groups_kept_frac', 'dynamic-sampling keep rate'),
                    ('gen_exec_match', 'dev execution match'), ('gen_exact_match', 'dev exact match')]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (key, title) in zip(axes.flat, metrics_to_plot):
    for seed in CFG['seeds']:
        hp = os.path.join(seed_out_dir(seed), 'history.json')
        if not os.path.exists(hp):
            continue
        h = json.load(open(hp))
        if not h[key]:
            continue
        steps = list(range(1, len(h[key]) + 1))
        ax.plot(steps, h[key], marker='.', linestyle='-', label=f'seed {seed}')
    ax.set_title(title); ax.set_xlabel('rollout step'); ax.legend(fontsize=8)
fig.tight_layout()
fig_path = os.path.join(OUT_BASE, 'rl_progress.png')
fig.savefig(fig_path, dpi=110)
print('saved ->', fig_path)
plt.show()

In [ ]:
## SFT-only vs SFT+RL: head-to-head execution efficiency (independent add-on)
#
# The existing gen_efficiency_speedup measures pred-vs-GOLD, and SFT-only has no value for it
# (added after sft_train.ipynb wrote its test_eval.json). This cell instead times the SFT model's
# own SQL against the RL model's own SQL, on identical rows, so the baseline is the SFT query --
# the actual "did RL make the SQL faster than SFT did" question.
#
# Only rows where BOTH models' SQL is exec-correct (matches gold result set, order-insensitive)
# are compared -- timing a wrong query is meaningless. Same cache-fair protocol as compute_reward:
# an untimed warm-up of both queries, then timed executions in randomized order, repeated and
# median-reduced to damp sub-millisecond measurement noise. Ratio = sft_time / rl_time, so >1 means
# RL is faster; a per-row median is taken first, then a median across rows.
#
# IMPORTANT caveat surfaced by the numbers: gen_exact_match is ~0.982, i.e. SFT and RL emit
# byte-identical SQL on almost every row. Those rows have ratio 1.0 by construction and carry no
# signal, so this cell counts and EXCLUDES them, reporting the efficiency delta only over rows where
# the two models actually produced different SQL -- and prints how few those are, which is the real
# story. Run on a large sample (full test_rows by default) or there will be nothing to compare.

EFF_TIMING_REPS = 7            # timed reps per query per row; odd so the median is a real datapoint
EFF_SAMPLE = test_rows         # full split by default -- at exact_match~0.98, small samples yield ~0 differing rows

def _norm_sql(s):
    return ' '.join(s.strip().split())   # collapse whitespace so trivially-formatted-differently == identical

@torch.no_grad()
def _greedy_preds(model, tokenizer, sample_rows, schema_ddl_, cfg):
    '''Same greedy decode as evaluate_generation, returned as a list of pred strings (one per row).
    Batched: multiple prompts per generate() call instead of one at a time -- same
    fix as evaluate_generation, applied here too since this cell has its own separate generation
    loop. Batch size reuses rollout_prompts_per_batch x group_size, the concurrency level the
    rollout batch-size probe already validated as safe (greedy eval has no group_size replication,
    so it's strictly lighter per call than what that probe tested).'''
    model.eval()
    prev = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    batch_size = cfg.get('eval_gen_batch_size', cfg['rollout_prompts_per_batch'] * cfg['group_size'])
    preds = [None] * len(sample_rows)
    for start in tqdm(range(0, len(sample_rows), batch_size), desc='generate', leave=False):
        batch_rows = sample_rows[start:start + batch_size]
        prompt_texts = []
        for r in batch_rows:
            messages = build_messages(r, schema_ddl_)[:-1]
            prompt_texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        enc = tokenizer(prompt_texts, return_tensors='pt', padding=True, truncation=True,
                         max_length=cfg['max_seq_len'], add_special_tokens=False).to(DEVICE)
        padded_prompt_len = enc['input_ids'].shape[1]
        gen = model.generate(**enc, max_new_tokens=cfg.get('max_new_tokens_eval', 2048),
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
        for i in range(len(batch_rows)):
            completion = tokenizer.decode(gen[i][padded_prompt_len:], skip_special_tokens=True).strip()
            preds[start + i] = extract_sql_from_completion(completion) or ''
    tokenizer.padding_side = prev
    return preds

def _pairwise_speedup(con, sft_sql, rl_sql, cfg, reps):
    '''Median over `reps` of (sft_time / rl_time), cache-fair: warm both once untimed, then time
    both per rep in randomized order. >1 => RL query faster. Floors at efficiency_min_timing_ms.'''
    floor_s = cfg['efficiency_min_timing_ms'] / 1000
    _exec_rows(con, sft_sql); _exec_rows(con, rl_sql)   # untimed warm-up, equalize buffer-pool state
    ratios = []
    for _ in range(reps):
        order = ['sft', 'rl'] if random.random() < 0.5 else ['rl', 'sft']
        t = {}
        for which in order:
            sql = sft_sql if which == 'sft' else rl_sql
            t0 = time.perf_counter(); _exec_rows(con, sql); t[which] = max(time.perf_counter() - t0, floor_s)
        ratios.append(t['sft'] / t['rl'])
    return statistics.median(ratios)

def compare_sft_rl_efficiency(seed, sample_rows, con, cfg):
    sft_best_dir = os.path.join(SFT_OUT_BASE, f'seed_{seed}', 'best')
    rl_best_dir = os.path.join(seed_out_dir(seed), 'best')
    if not (os.path.isdir(sft_best_dir) and os.path.isdir(rl_best_dir)):
        print(f'[skip] seed {seed}: missing SFT or RL best/ checkpoint'); return None

    # Base loaded once; both adapters live on it and we switch with set_adapter (same pattern as
    # evaluate_checkpoint_on_test) -- avoids reloading the 14B base twice per seed.
    wrapper = FHIRSQLLLM(cfg)
    tokenizer = wrapper._load_tokenizer()
    base_model = wrapper._load_base_model()
    model = PeftModel.from_pretrained(base_model, sft_best_dir, adapter_name='sft', is_trainable=False)
    model.load_adapter(rl_best_dir, adapter_name='rl')
    model.to(DEVICE)

    model.set_adapter('sft'); sft_preds = _greedy_preds(model, tokenizer, sample_rows, schema_ddl, cfg)
    model.set_adapter('rl');  rl_preds  = _greedy_preds(model, tokenizer, sample_rows, schema_ddl, cfg)

    n_answerable = n_both_correct = n_identical = 0
    speedups = []
    for r, sft_p, rl_p in zip(sample_rows, sft_preds, rl_preds):
        gold = r['gold_sql'].strip()
        if gold == ABSTENTION_TOKEN or sft_p == ABSTENTION_TOKEN or rl_p == ABSTENTION_TOKEN:
            continue
        n_answerable += 1
        try:
            gold_rows = _exec_rows(con, gold)
            if _exec_rows(con, sft_p) != gold_rows or _exec_rows(con, rl_p) != gold_rows:
                continue   # only compare where BOTH are correct
        except Exception:
            continue
        n_both_correct += 1
        if _norm_sql(sft_p) == _norm_sql(rl_p):
            n_identical += 1   # byte-identical SQL -> ratio 1.0 by construction, no signal; excluded
            continue
        speedups.append(_pairwise_speedup(con, sft_p, rl_p, cfg, EFF_TIMING_REPS))

    del model, base_model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

    return {
        'n_answerable': n_answerable, 'n_both_correct': n_both_correct,
        'n_identical_sql': n_identical, 'n_differing_compared': len(speedups),
        'median_rl_speedup_over_sft': (statistics.median(speedups) if speedups else float('nan')),
        'mean_rl_speedup_over_sft': (statistics.mean(speedups) if speedups else float('nan')),
        'frac_rl_faster': (sum(s > 1 for s in speedups) / len(speedups) if speedups else float('nan')),
    }

print(f'sample = {len(EFF_SAMPLE)} rows | {EFF_TIMING_REPS} timing reps/query | ratio>1 => RL faster\n')
eff_results = {}
for seed in CFG['seeds']:
    print(f'[seed {seed}] SFT-vs-RL efficiency head-to-head starting...')
    _eff_t0 = time.time()
    res = compare_sft_rl_efficiency(seed, EFF_SAMPLE, duckdb_con, CFG)
    if res is None:
        continue
    eff_results[seed] = res
    print(f'[seed {seed}] done in {(time.time()-_eff_t0)/60:.1f} min')
    print(f'seed {seed}: both-correct {res["n_both_correct"]}/{res["n_answerable"]} answerable '
          f'| identical SQL {res["n_identical_sql"]} | differing & compared {res["n_differing_compared"]}')
    if res['n_differing_compared']:
        print(f'         median RL-vs-SFT speedup {res["median_rl_speedup_over_sft"]:.3f} '
              f'| mean {res["mean_rl_speedup_over_sft"]:.3f} '
              f'| RL faster on {res["frac_rl_faster"]:.0%} of differing rows')
    else:
        print('         no rows where both models were correct AND produced different SQL '
              '-- nothing to compare (expected, given exact-match ~0.98)')
